In [ ]:
# ============================================================
# SYNTHETIC CROWD GENERATOR V7
# ============================================================
#
# COMBINES
# ------------------------------------------------------------
# ✔ Explicit temporal phases (bottleneck + panic)
# ✔ Expanding panic wave from persistent origin
# ✔ Stress memory accumulation per agent
# ✔ Stop-go bottleneck oscillations
# ✔ Queue-front compression propagation
# ✔ post_warmup_factor ramp (Fix 4) — weakens early signal
#   so LSTM must accumulate evidence over time
# ✔ Per-sequence phase offset for normal sequences
# ✔ Group boundary reflection fix
# ✔ Group index stored (not reference)
# ✔ panic_origin only for panic class
# ✔ Bounded global flow magnitudes
# ✔ Identical agent priors at spawn
# ✔ 200 sequences per class
#
# ============================================================

import os
import cv2
import numpy as np
import scipy.ndimage
from scipy.spatial import KDTree

# ============================================================
# CONFIG
# ============================================================

IMG_SIZE       = 512
FRAMES_PER_SEQ = 30

NUM_NORMAL     = 200
NUM_BOTTLENECK = 200
NUM_PANIC      = 200

BASE_DIR      = "synthetic_crowd_v7"
WARMUP_FRAMES = 5

np.random.seed(42)

CLASS_NAMES = {
    0: "0_normal",
    1: "1_bottleneck",
    2: "2_panic"
}

for c in CLASS_NAMES.values():
    os.makedirs(os.path.join(BASE_DIR, c), exist_ok=True)

# ============================================================
# GEOMETRY
# ============================================================

def random_geometry():

    wall_y         = np.random.randint(380, 440)
    wall_thickness = np.random.randint(50, 90)
    gate_width     = np.random.randint(40, 80)
    gate_x         = np.random.randint(80, IMG_SIZE - 80 - gate_width)

    return {
        "wall_y_min":  wall_y,
        "wall_y_max":  wall_y + wall_thickness,
        "gate_x_min":  gate_x,
        "gate_x_max":  gate_x + gate_width,
        "gate_center": np.array([
            gate_x + gate_width / 2,
            wall_y + wall_thickness / 2
        ], dtype=np.float32)
    }

# ============================================================
# COLLISION
# ============================================================

def circle_rect_collision(cx, cy, r, rx1, ry1, rx2, ry2):
    nearest_x = np.clip(cx, rx1, rx2)
    nearest_y = np.clip(cy, ry1, ry2)
    dx = cx - nearest_x
    dy = cy - nearest_y
    return (dx * dx + dy * dy) < (r * r)

def wall_collision(pos, r, g):
    x, y = pos
    if circle_rect_collision(x, y, r, 0, g["wall_y_min"],
                             g["gate_x_min"], g["wall_y_max"]):
        return True
    if circle_rect_collision(x, y, r, g["gate_x_max"], g["wall_y_min"],
                             IMG_SIZE, g["wall_y_max"]):
        return True
    return False

# ============================================================
# GROUP
# ============================================================

class CrowdGroup:

    def __init__(self, geometry):

        self.center = np.array([
            np.random.uniform(50, IMG_SIZE - 50),
            np.random.uniform(50, geometry["wall_y_min"] - 50)
        ], dtype=np.float32)

        self.velocity = np.random.randn(2).astype(np.float32)
        self.velocity /= (np.linalg.norm(self.velocity) + 1e-6)

        self.coherence = np.random.uniform(0.3, 1.0)
        self.radius    = np.random.uniform(40, 120)
        self.timer     = np.random.randint(20, 80)

    def update(self, geometry):

        self.timer -= 1

        if self.timer <= 0:
            self.velocity = np.random.randn(2).astype(np.float32)
            self.velocity /= (np.linalg.norm(self.velocity) + 1e-6)
            self.coherence = np.random.uniform(0.3, 1.0)
            self.timer     = np.random.randint(20, 80)

        # Reflect on boundary contact so groups don't
        # get stuck at walls with outward-pointing velocity
        new_center = self.center + self.velocity * 1.5
        if new_center[0] < 40 or new_center[0] > IMG_SIZE - 40:
            self.velocity[0] *= -1
        if new_center[1] < 40 or new_center[1] > geometry["wall_y_min"] - 40:
            self.velocity[1] *= -1

        self.center += self.velocity * 1.5
        self.center[0] = np.clip(self.center[0], 40, IMG_SIZE - 40)
        self.center[1] = np.clip(self.center[1], 40,
                                  geometry["wall_y_min"] - 40)

# ============================================================
# AGENT
# ============================================================

class Agent:

    def __init__(self, geometry, num_groups):

        self.r = np.random.uniform(5, 9)

        while True:
            x = np.random.uniform(self.r, IMG_SIZE - self.r)
            y = np.random.uniform(self.r, geometry["wall_y_min"] - 60)
            if not wall_collision((x, y), self.r, geometry):
                break

        self.pos  = np.array([x, y], dtype=np.float32)
        self.prev = self.pos.copy()
        self.vel  = np.zeros(2, dtype=np.float32)

        # Store index, not reference — avoids stale reference bugs
        self.group_idx   = np.random.randint(0, num_groups)
        self.group_timer = np.random.randint(20, 60)

        # Identical priors at spawn — no class signal before warmup
        self.goal_strength   = np.random.uniform(0.8, 1.5)
        self.noise_strength  = np.random.uniform(0.3, 0.8)
        self.social_strength = np.random.uniform(0.5, 1.5)

        self.role = np.random.choice(
            ["gate", "wander", "group"],
            p=[0.4, 0.3, 0.3]
        )

        self.local_target = np.array([
            np.random.uniform(0, IMG_SIZE),
            np.random.uniform(0, geometry["wall_y_min"])
        ], dtype=np.float32)

        # Persistent internal state — accumulates from crowd pressure
        # Starts at a small random value identical across all classes
        self.stress  = np.random.uniform(0.0, 0.2)
        self.fatigue = 0.0

# ============================================================
# TEMPORAL PHASES
# ============================================================

def get_bottleneck_phase(t):
    """
    Maps frame index to a named bottleneck phase.
    Phases are irreversible — crowd cannot un-congest.
    This creates a causal temporal signature that temporal
    shuffle destroys, forcing the LSTM to use ordering.
    """
    if   t < 10: return "converging"
    elif t < 17: return "compression"
    elif t < 23: return "stop_go"
    else:        return "queue_lock"

def get_panic_phase(t):
    """
    Maps frame index to a named panic phase.
    Panic escalates irreversibly from localised disturbance
    to global fragmentation over the sequence.
    """
    if   t < 10: return "disturbance"
    elif t < 15: return "localized_escape"
    elif t < 20: return "propagation"
    elif t < 25: return "fragmentation"
    else:        return "global_panic"

# ============================================================
# GLOBAL FLOW FIELD
# ============================================================

def compute_global_flow(effective_class, geometry, t, seq_phase):
    """
    Scene-level flow field that biases all agents.
    Each class has a distinct temporal evolution.
    seq_phase ensures normal sequences are not all identical.
    """

    temporal_factor = t / FRAMES_PER_SEQ

    # post_warmup_factor: 0.0 at frame 5, 1.0 at frame 29
    # Used for class-specific scaling so early post-warmup
    # frames are still ambiguous and the class signal builds
    # gradually — forcing the LSTM to accumulate evidence
    post_warmup = max(0.0,
        (t - WARMUP_FRAMES) / (FRAMES_PER_SEQ - WARMUP_FRAMES)
    )

    if effective_class == 0:

        # Normal: gentle oscillating field, unique phase per sequence
        angle = np.sin(temporal_factor * np.pi * 2 + seq_phase) * 0.5
        return np.array([np.cos(angle), np.sin(angle)], dtype=np.float32)

    elif effective_class == 1:

        # Bottleneck: convergence toward gate grows quadratically
        # with post_warmup so early frames are still ambiguous
        gate_dir = geometry["gate_center"] - np.array(
            [IMG_SIZE / 2, geometry["wall_y_min"] / 2]
        )
        gate_dir /= (np.linalg.norm(gate_dir) + 1e-6)
        strength = post_warmup ** 2   # quadratic ramp from warmup end
        return gate_dir * strength

    else:

        # Panic: random direction each frame, magnitude grows
        # Capped at 1.5 — previous 2.5 caused all agents to
        # hit speed limit every frame, making speed a trivial feature
        angle     = np.random.uniform(0, np.pi * 2)
        magnitude = np.clip(0.3 + post_warmup * 1.2, 0.3, 1.5)
        return np.array(
            [np.cos(angle), np.sin(angle)], dtype=np.float32
        ) * magnitude

# ============================================================
# UPDATE AGENTS
# ============================================================

def update_agents(
    agents, groups, geometry,
    class_type, t,
    panic_origin, seq_phase
):
    temporal_factor = t / FRAMES_PER_SEQ

    # post_warmup_factor: 0 at frame 5, 1 at frame 29
    # Scales all class-specific forces so they build gradually
    # after warmup ends rather than switching on abruptly
    post_warmup = max(0.0,
        (t - WARMUP_FRAMES) / (FRAMES_PER_SEQ - WARMUP_FRAMES)
    )

    # Warmup: all classes identical
    effective_class = 0 if t < WARMUP_FRAMES else class_type

    # Determine phase names (only used for the relevant class)
    bottleneck_phase = get_bottleneck_phase(t) if effective_class == 1 else None
    panic_phase      = get_panic_phase(t)      if effective_class == 2 else None

    # Panic wave radius expands with phase
    panic_radius_map = {
        "disturbance":     40,
        "localized_escape": 80,
        "propagation":     140,
        "fragmentation":   220,
        "global_panic":    400
    }
    panic_radius = panic_radius_map.get(panic_phase, 0)

    for g in groups:
        g.update(geometry)

    positions = np.array([a.pos for a in agents])
    tree      = KDTree(positions)

    global_flow = compute_global_flow(
        effective_class, geometry, t, seq_phase
    )

    for a in agents:

        a.prev = a.pos.copy()

        # Group switching
        a.group_timer -= 1
        if a.group_timer <= 0:
            a.group_idx   = np.random.randint(0, len(groups))
            a.group_timer = np.random.randint(20, 60)
        current_group = groups[a.group_idx]

        # Role target
        if a.role == "gate":
            target = geometry["gate_center"]
        elif a.role == "wander":
            if np.random.rand() < 0.02:
                a.local_target = np.array([
                    np.random.uniform(0, IMG_SIZE),
                    np.random.uniform(0, geometry["wall_y_min"])
                ], dtype=np.float32)
            target = a.local_target
        else:
            target = current_group.center

        # Goal force
        goal = target - a.pos
        goal /= (np.linalg.norm(goal) + 1e-6)

        # Neighbors
        neighbor_idx  = tree.query_ball_point(a.pos, 35)
        local_density = max(0, len(neighbor_idx) - 1)

        # Repulsion
        repulsion = np.zeros(2, dtype=np.float32)
        for idx in neighbor_idx:
            other = agents[idx]
            if other is a:
                continue
            delta   = a.pos - other.pos
            dist    = np.linalg.norm(delta) + 1e-6
            overlap = (a.r + other.r) - dist
            if overlap > 0:
                repulsion += (delta / dist) * overlap * 1.0

        # Alignment with neighbours
        alignment = np.zeros(2, dtype=np.float32)
        if len(neighbor_idx) > 1:
            for idx in neighbor_idx:
                other = agents[idx]
                if other is not a:
                    alignment += other.vel
            alignment /= (len(neighbor_idx) - 1)

        # Group coherence — scaled by per-agent social_strength
        group_force  = current_group.center - a.pos
        group_force /= (np.linalg.norm(group_force) + 1e-6)
        group_force *= current_group.coherence * a.social_strength

        # Local pressure
        pressure = local_density / 10.0

        # Stress memory — accumulates from crowd pressure, decays slowly
        # This gives agents history that changes their behaviour over time
        a.stress += pressure * 0.01
        a.stress *= 0.995
        a.stress  = np.clip(a.stress, 0, 3)

        # Panic contagion from persistent origin point
        # Only active for panic class, only if agent is inside panic_radius
        panic_force = np.zeros(2, dtype=np.float32)
        if effective_class == 2 and panic_origin is not None and panic_radius > 0:
            delta = a.pos - panic_origin
            dist  = np.linalg.norm(delta) + 1e-6
            if dist < panic_radius:
                panic_force += (delta / dist) * (panic_radius - dist) * 0.03
                a.stress    += 0.03

        # ====================================================
        # CLASS + PHASE SPECIFIC DYNAMICS
        # All forces scaled by post_warmup so they ramp in
        # gradually after warmup ends rather than abruptly.
        # This is Fix 4 — makes early windows ambiguous and
        # forces the LSTM to accumulate temporal evidence.
        # ====================================================

        if effective_class == 0:

            goal_scale   = 0.6
            align_scale  = 0.5
            noise_scale  = 0.4 + 0.2 * post_warmup
            global_scale = 0.4
            speed_limit  = 2.0 + np.random.uniform(-0.2, 0.2)

        elif effective_class == 1:

            if bottleneck_phase == "converging":
                goal_scale   = (0.5 + 0.5 * post_warmup)
                align_scale  = min(0.8, 0.4 + 0.6 * post_warmup)
                noise_scale  = 0.5
                global_scale = (0.3 + 0.7 * post_warmup)
                speed_limit  = 2.2

            elif bottleneck_phase == "compression":
                goal_scale   = (0.8 + 0.5 * post_warmup)
                align_scale  = min(1.0, 0.6 + 0.6 * post_warmup)
                noise_scale  = 0.3
                global_scale = (0.7 + 0.6 * post_warmup)
                speed_limit  = max(0.5, 1.8 - pressure * 0.3)

            elif bottleneck_phase == "stop_go":
                oscillation  = 0.7 + 0.3 * np.sin(t * 0.8)
                goal_scale   = 1.2
                align_scale  = 1.0
                noise_scale  = 0.3
                global_scale = 1.5
                speed_limit  = max(0.3, 2.2 * oscillation - pressure * 0.2)

            else:  # queue_lock
                goal_scale   = 1.5
                align_scale  = min(1.3, 1.0 + 0.3 * post_warmup)
                noise_scale  = 0.2
                global_scale = 1.8
                speed_limit  = max(0.3, 1.0 - pressure * 0.1)

        else:  # panic

            if panic_phase == "disturbance":
                noise_scale  = 0.5 + 0.3 * post_warmup
                global_scale = 0.3 + 0.3 * post_warmup
                align_scale  = 0.5
                speed_limit  = 2.0 + 0.5 * post_warmup

            elif panic_phase == "localized_escape":
                noise_scale  = 1.0 + 0.3 * post_warmup
                global_scale = 0.8 + 0.2 * post_warmup
                align_scale  = 0.3
                speed_limit  = 2.8 + 0.4 * post_warmup

            elif panic_phase == "propagation":
                noise_scale  = 1.5 + 0.3 * post_warmup
                global_scale = 1.2 + 0.3 * post_warmup
                align_scale  = 0.2
                speed_limit  = 3.5 + 0.3 * post_warmup

            elif panic_phase == "fragmentation":
                noise_scale  = 2.0 + 0.2 * post_warmup
                global_scale = 1.6 + 0.2 * post_warmup
                align_scale  = 0.1
                speed_limit  = 4.0 + 0.2 * post_warmup

            else:  # global_panic
                noise_scale  = 2.5
                global_scale = 2.0
                align_scale  = 0.05
                speed_limit  = min(5.0, 4.5 + pressure * 0.1)

            goal_scale = 0.3

        turbulence = np.random.randn(2) * noise_scale

        force = (
            goal        * goal_scale
            + repulsion  * 2.5
            + alignment  * align_scale
            + group_force                  # already scaled by social_strength
            + global_flow * global_scale
            + turbulence
            + panic_force
        )

        # Queue front compression — slows agents near gate
        # in bottleneck, scaled by post_warmup so effect builds
        if effective_class == 1:
            dist_gate = np.linalg.norm(a.pos - geometry["gate_center"])
            if dist_gate < 140:
                compression = pressure * post_warmup
                force *= (1.0 - compression * 0.2)

        # Speed cap
        speed = np.linalg.norm(force)
        if speed > speed_limit:
            force = (force / speed) * speed_limit

        proposed    = a.pos + force
        proposed[0] = np.clip(proposed[0], a.r, IMG_SIZE - a.r)
        proposed[1] = np.clip(proposed[1], a.r, IMG_SIZE - a.r)

        if not wall_collision(proposed, a.r, geometry):
            a.vel = proposed - a.pos
            a.pos = proposed
        else:
            a.vel *= -0.2

# ============================================================
# DENSITY MAP
# ============================================================

def generate_density(agents):
    density = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for a in agents:
        x = int(a.pos[0])
        y = int(a.pos[1])
        if 0 <= x < IMG_SIZE and 0 <= y < IMG_SIZE:
            density[y, x] += 1.0
    return scipy.ndimage.gaussian_filter(density, sigma=5)

# ============================================================
# FLOW MAP
# ============================================================

def generate_flow(agents):
    flow = np.zeros((IMG_SIZE, IMG_SIZE, 2), dtype=np.float32)
    for a in agents:
        x = int(a.pos[0])
        y = int(a.pos[1])
        if 0 <= x < IMG_SIZE and 0 <= y < IMG_SIZE:
            flow[y, x, 0] = a.vel[0]
            flow[y, x, 1] = a.vel[1]
    return flow

# ============================================================
# RENDER
# ============================================================

def render_frame(agents, geometry):

    bg  = np.random.randint(235, 256)
    img = np.ones((IMG_SIZE, IMG_SIZE), dtype=np.uint8) * bg

    wall_color = np.random.randint(90, 160)

    cv2.rectangle(img, (0, geometry["wall_y_min"]),
                  (geometry["gate_x_min"], geometry["wall_y_max"]),
                  wall_color, -1)
    cv2.rectangle(img, (geometry["gate_x_max"], geometry["wall_y_min"]),
                  (IMG_SIZE, geometry["wall_y_max"]),
                  wall_color, -1)

    for a in agents:
        intensity = np.random.randint(0, 50)
        cv2.circle(img, (int(a.pos[0]), int(a.pos[1])),
                   int(a.r), intensity, -1)

    if np.random.rand() < 0.3:
        k   = np.random.choice([3, 5])
        img = cv2.GaussianBlur(img, (k, k), 0)

    if np.random.rand() < 0.3:
        noise = np.random.normal(0, 5, img.shape)
        img   = np.clip(img + noise, 0, 255).astype(np.uint8)

    return img

# ============================================================
# SEQUENCE
# ============================================================

def generate_sequence(seq_id, class_type):

    geometry = random_geometry()

    seq_dir = os.path.join(
        BASE_DIR, CLASS_NAMES[class_type], f"seq_{seq_id:04d}"
    )
    os.makedirs(seq_dir, exist_ok=True)

    num_agents = np.random.randint(80, 160)
    num_groups = np.random.randint(4, 10)

    groups = [CrowdGroup(geometry) for _ in range(num_groups)]
    agents = [Agent(geometry, num_groups) for _ in range(num_agents)]

    # Panic origin: fixed per sequence, only meaningful for panic class
    # Passed to update_agents for all classes but only applied
    # when effective_class == 2
    panic_origin = np.array([
        np.random.uniform(100, IMG_SIZE - 100),
        np.random.uniform(100, geometry["wall_y_min"] - 100)
    ], dtype=np.float32) if class_type == 2 else None

    # Per-sequence phase offset for normal flow field
    # Ensures no two normal sequences have identical flow directions
    seq_phase = np.random.uniform(0, np.pi * 2)

    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    mask[
        geometry["wall_y_min"]:geometry["wall_y_max"],
        geometry["gate_x_min"]:geometry["gate_x_max"]
    ] = 1
    np.save(os.path.join(seq_dir, "mask.npy"), mask)

    for t in range(FRAMES_PER_SEQ):

        update_agents(
            agents, groups, geometry,
            class_type, t,
            panic_origin, seq_phase
        )

        frame   = render_frame(agents, geometry)
        density = generate_density(agents)
        flow    = generate_flow(agents)

        cv2.imwrite(os.path.join(seq_dir, f"frame_{t:03d}.png"), frame)
        np.save(os.path.join(seq_dir, f"density_{t:03d}.npy"), density)
        np.save(os.path.join(seq_dir, f"flow_{t:03d}.npy"), flow)

# ============================================================
# DRIVER
# ============================================================

seq_id = 0

print("Generating NORMAL sequences...")
for _ in range(NUM_NORMAL):
    generate_sequence(seq_id, 0)
    seq_id += 1

print("Generating BOTTLENECK sequences...")
for _ in range(NUM_BOTTLENECK):
    generate_sequence(seq_id, 1)
    seq_id += 1

print("Generating PANIC sequences...")
for _ in range(NUM_PANIC):
    generate_sequence(seq_id, 2)
    seq_id += 1

print("=" * 60)
print("SYNTHETIC DATASET GENERATION COMPLETE")
print("=" * 60)

In [ ]:
#revised data split
import os
import shutil
import random

BASE_DIR = "synthetic_crowd_v7"
SPLIT_BASE = "split_dataset_v7"
CLASSES = ["0_normal", "1_bottleneck", "2_panic"]

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

random.seed(42)

if os.path.exists(SPLIT_BASE):
    print(f"Removing old split: {SPLIT_BASE}")
    shutil.rmtree(SPLIT_BASE)

for split in ['train', 'val', 'test']:
    for c in CLASSES:
        os.makedirs(os.path.join(SPLIT_BASE, split, c), exist_ok=True)

print("\nSplitting dataset...")

for c in CLASSES:

    class_path = os.path.join(BASE_DIR, c)

    sequences = [
        s for s in sorted(os.listdir(class_path))
        if os.path.isdir(os.path.join(class_path, s))
    ]

    random.shuffle(sequences)

    n_train = int(len(sequences) * TRAIN_RATIO)
    n_val   = int(len(sequences) * VAL_RATIO)

    splits = {
        'train': sequences[:n_train],
        'val':   sequences[n_train : n_train + n_val],
        'test':  sequences[n_train + n_val:]
    }

    for split_name, seq_list in splits.items():
        for seq in seq_list:
            src = os.path.join(class_path, seq)
            # FIX: c and split_name are used directly here,
            # not captured inside a nested function.
            dst = os.path.join(SPLIT_BASE, split_name, c, seq)
            shutil.copytree(src, dst)

    print(f"\nClass : {c}")
    print(f"  Total : {len(sequences)}")
    print(f"  Train : {len(splits['train'])}")
    print(f"  Val   : {len(splits['val'])}")
    print(f"  Test  : {len(splits['test'])}")

print("\n" + "="*60)
print("SPLIT COMPLETE")
print("="*60)

In [1]:
# ============================================================
# CELL 1 : IMPORTS + DEVICE SETUP
# ============================================================

import os
import glob
import cv2
import numpy as np

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from tqdm.auto import tqdm

# ============================================================
# REPRODUCIBILITY
# ============================================================

torch.manual_seed(42)

np.random.seed(42)

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 60)
print(f"DEVICE : {device}")
print("=" * 60)

# ============================================================
# AMP DISABLED
# ============================================================
#
# AMP caused runtime instability on current setup.
# We disable it completely for stable debugging.
#
# ============================================================

USE_AMP = False

DEVICE : cuda


C:\Users\RAJESH\csrnet-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(device)
print(USE_AMP)
print(torch.cuda.is_available())

cuda
False
True


In [2]:
# ============================================================
# CELL 2 : DATASET WITH AUGMENTATION PIPELINE
# ============================================================

import os
import glob
import cv2
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

WARMUP_FRAMES   = 5
DENSITY_OUT_RES = 28
SCALE_FACTOR    = (512 * 512) / (DENSITY_OUT_RES * DENSITY_OUT_RES)

# ============================================================
# FRAME AUGMENTATION
# ============================================================

def augment_frames(frames_np, aug_prob=0.5):
    """
    Applies random photometric augmentation to a list of
    uint8 numpy arrays (H, W) during training.

    Augmentations applied independently per sequence with
    probability aug_prob. Within a sequence, the same
    augmentation type and parameters are applied to ALL frames
    to preserve temporal consistency — otherwise the model
    would see unnatural frame-to-frame contrast changes.

    frames_np : list of (H,W) uint8 arrays
    returns   : list of (H,W) uint8 arrays
    """

    if np.random.rand() > aug_prob:
        return frames_np   # no augmentation this sequence

    mode = np.random.choice([
    "gaussian",
    "blur",
    "brightness",
    "salt_pepper",
    "fog",
    "frame_dropout",
    "none"
], p=[0.17, 0.14, 0.14, 0.18, 0.14, 0.07, 0.16])

    if mode == "none":
        return frames_np

    augmented = []

    if mode == "gaussian":
        # Thermal sensor noise — sigma in pixel units [0,255]
        sigma = np.random.uniform(5, 20)
        for f in frames_np:
            noise = np.random.normal(0, sigma, f.shape).astype(np.float32)
            aug = np.clip(f.astype(np.float32) + noise, 0, 255).astype(np.uint8)
            augmented.append(aug)

    elif mode == "blur":
        # Motion blur or defocus — kernel size 3,5,7,9
        k = np.random.choice([3, 5, 7, 9])
        for f in frames_np:
            augmented.append(cv2.GaussianBlur(f, (k, k), 0))

    elif mode == "brightness":
        # Illumination change — multiplicative scale
        factor = np.random.uniform(0.5, 1.6)
        for f in frames_np:
            aug = np.clip(f.astype(np.float32) * factor, 0, 255).astype(np.uint8)
            augmented.append(aug)

    elif mode == "salt_pepper":
        # Dead / hot pixels
        prob = np.random.uniform(0.005, 0.05)
        for f in frames_np:
            aug = f.copy()
            mask = np.random.rand(*f.shape)
            aug[mask < prob / 2] = 0
            aug[(mask >= prob / 2) & (mask < prob)] = 255
            augmented.append(aug)

    elif mode == "fog":
        # Contrast attenuation toward mid-grey
        alpha = np.random.uniform(0.15, 0.50)
        for f in frames_np:
            aug = np.clip(
                (1 - alpha) * f.astype(np.float32) + alpha * 128,
                0, 255
            ).astype(np.uint8)
            augmented.append(aug)

    #return augmented

    elif mode == "frame_dropout":

    # Simulates packet/frame transmission loss
    # Important for ConvLSTM temporal robustness

        augmented = [f.copy() for f in frames_np]

    # Drop 1 or 2 frames max
        n_drop = np.random.choice([1, 2], p=[0.8, 0.2])

        T = len(augmented)

        drop_indices = np.random.choice(
            np.arange(T),
            size=n_drop,
            replace=False)

        for idx in drop_indices:

        # Option 1: complete blackout
            if np.random.rand() < 0.5:
                augmented[idx][:] = 0

        # Option 2: frozen previous frame
            else:
                if idx > 0:
                    augmented[idx] = augmented[idx - 1].copy()

    return augmented


# ============================================================
# FLOW AUGMENTATION
# ============================================================

def augment_flow(flow_tensor, training=True):
    """
    Applies random augmentation to flow tensors.
    Input/output: (T, 2, H, W) float tensor.

    1. Random magnitude scale [0.8, 1.2]
    2. Random horizontal flip (50%) with x-channel negation
    3. Random mild additive noise (20% of sessions)
       simulates Farneback estimation error
    """
    if not training:
        return flow_tensor

    # Magnitude scale
    scale = np.random.uniform(0.8, 1.2)
    flow_tensor = flow_tensor * scale

    # Horizontal flip
    if np.random.rand() < 0.5:
        flow_tensor = torch.flip(flow_tensor, dims=[3])
        flow_tensor[:, 0, :, :] = -flow_tensor[:, 0, :, :]

    # Mild additive noise (simulates Farneback estimation error)
    if np.random.rand() < 0.1:
        noise_sigma = np.random.uniform(0.03, 0.06)
    else:
        noise_sigma = np.random.uniform(0.005, 0.02)
    flow_tensor = flow_tensor + torch.randn_like(flow_tensor) * noise_sigma

    return torch.clamp(flow_tensor, -1.0, 1.0)


# ============================================================
# DATASET
# ============================================================

class CrowdDataset(Dataset):

    def __init__(
        self,
        base_dir,
        seq_length=15,
        stride=3,
        training=False,
        aug_prob=0.5     # probability of applying frame augmentation
    ):
        self.seq_length = seq_length
        self.training   = training
        self.aug_prob   = aug_prob
        self.windows    = []
        self.classes    = {
            "0_normal":     0,
            "1_bottleneck": 1,
            "2_panic":      2
        }

        print("=" * 60)
        print(f"LOADING DATASET : {base_dir}")
        print(f"seq_length      : {seq_length}")
        print(f"stride          : {stride}")
        print(f"training mode   : {training}")
        print(f"aug_prob        : {aug_prob if training else 'N/A (val/test)'}")
        print("=" * 60)

        for c_name, c_idx in self.classes.items():
            class_dir = os.path.join(base_dir, c_name)
            if not os.path.exists(class_dir):
                continue
            for seq_folder in sorted(os.listdir(class_dir)):
                seq_path = os.path.join(class_dir, seq_folder)
                if not os.path.isdir(seq_path):
                    continue
                frame_files = sorted(
                    glob.glob(os.path.join(seq_path, "frame_*.png"))
                )
                num_frames = len(frame_files)
                if num_frames < seq_length + 1:
                    continue
                for start_idx in range(
                    WARMUP_FRAMES,
                    num_frames - seq_length,
                    stride
                ):
                    self.windows.append({
                        "seq_path":  seq_path,
                        "label":     c_idx,
                        "start_idx": start_idx
                    })

        print(f"TOTAL WINDOWS : {len(self.windows)}")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):

        w        = self.windows[idx]
        seq_path = w["seq_path"]
        label    = w["label"]
        start    = w["start_idx"]

        # ── Load raw frames ──────────────────────────────────
        raw_uint8 = []
        for t in range(start, start + self.seq_length + 1):
            img_path = os.path.join(seq_path, f"frame_{t:03d}.png")
            img = Image.open(img_path).convert("L")
            img = img.resize((224, 224))
            raw_uint8.append(np.array(img))   # uint8 [0,255]

        # ── Frame augmentation (training only) ───────────────
        # Applied BEFORE Farneback so augmented frames
        # produce augmented flow — this is the key change.
        # The model trains on degraded flow, building
        # robustness to all the tested corruption types.
        if self.training:
            raw_uint8 = augment_frames(raw_uint8, aug_prob=self.aug_prob)

        # ── Farneback optical flow ───────────────────────────
        flows = []
        for t in range(self.seq_length):
            flow = cv2.calcOpticalFlowFarneback(
                raw_uint8[t],
                raw_uint8[t + 1],
                None,
                pyr_scale=0.5,
                levels=3,
                winsize=15,
                iterations=3,
                poly_n=5,
                poly_sigma=1.2,
                flags=0
            )
            flow = np.clip(flow / 8.0, -1.0, 1.0)
            # After: flow = np.clip(flow / 8.0, -1.0, 1.0)
            # Add flow spike suppression:
            # S&P noise creates isolated ±1.0 spikes in flow.
            # A 3x3 median filter removes isolated spikes while
            # preserving genuine motion boundaries.
            # Applied only probabilistically during training to
            # avoid making the model over-reliant on clean flow.
            if self.training and np.random.rand() < 0.3:
                flow_x = cv2.medianBlur(flow[:,:,0], 3)
                flow_y = cv2.medianBlur(flow[:,:,1], 3)
                flow   = np.stack([flow_x, flow_y], axis=2)

            flows.append(
                torch.tensor(flow, dtype=torch.float32).permute(2, 0, 1)
            )

        # ── Frame tensors ────────────────────────────────────
        frames = [
            torch.tensor(
                raw_uint8[t].astype(np.float32) / 255.0,
                dtype=torch.float32
            ).unsqueeze(0)
            for t in range(self.seq_length)
        ]

        frame_tensor = torch.stack(frames)   # (T, 1, 224, 224)
        flow_tensor  = torch.stack(flows)    # (T, 2, 224, 224)

        # ── Flow augmentation (training only) ────────────────
        if self.training:
            flow_tensor = augment_flow(flow_tensor, training=True)

        # ── Density target ───────────────────────────────────
        target_idx   = start + self.seq_length - 1
        density_path = os.path.join(
            seq_path, f"density_{target_idx:03d}.npy"
        )
        density_512 = np.load(density_path).astype(np.float32)
        density_28  = cv2.resize(
            density_512, (DENSITY_OUT_RES, DENSITY_OUT_RES),
            interpolation=cv2.INTER_AREA
        )
        density_28     = density_28 * SCALE_FACTOR
        density_tensor = torch.tensor(density_28, dtype=torch.float32).unsqueeze(0)

        # ── Verification on first item ────────────────────────
        if idx == 0:
            print(
                f"\nFLOW VERIFICATION (idx=0)\n"
                f"  flow std  : {flow_tensor.std():.4f}\n"
                f"  density sum : {density_tensor.sum():.1f} agents\n"
            )

        return (
            frame_tensor,
            flow_tensor,
            torch.tensor(label, dtype=torch.long),
            density_tensor
        )

In [3]:
# ============================================================
# CELL 3 : MODEL WITH MOTIONCNN FLOW BRANCH
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import os

# ============================================================
# CONVLSTM CELL
# ============================================================

class ConvLSTMCell(nn.Module):

    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size // 2
        self.conv = nn.Conv2d(
            input_dim + hidden_dim,
            4 * hidden_dim,
            kernel_size,
            padding=padding
        )
        self.bn = nn.BatchNorm2d(4 * hidden_dim)

    def forward(self, x, states):
        h_cur, c_cur = states
        combined = torch.cat([x, h_cur], dim=1)
        gates    = self.bn(self.conv(combined))
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


# ============================================================
# MOTIONCNN — deep flow feature extractor
# ============================================================

class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation channel attention.
    Learns to weight which flow channels are most informative
    for each spatial location — identical mechanism to the
    SqueezeExcitation blocks inside MobileNetV3.
    """
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4)),
            nn.ReLU(),
            nn.Linear(max(channels // reduction, 4), channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        B, C, _, _ = x.shape
        w = self.pool(x).view(B, C)
        w = self.fc(w).view(B, C, 1, 1)
        return x * w


class MotionResBlock(nn.Module):
    """
    Residual block for the flow branch.
    Depthwise separable convolution + SE attention + residual.
    Mirrors the InvertedResidual structure of MobileNetV3
    so the flow branch has comparable capacity to the
    spatial branch despite starting from only 2 channels.
    """
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.stride = stride

        # Pointwise expand
        mid_ch = out_ch * 2
        self.pw1 = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, 1, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.Hardswish()
        )
        # Depthwise spatial
        self.dw = nn.Sequential(
            nn.Conv2d(mid_ch, mid_ch, 3, stride=stride,
                      padding=1, groups=mid_ch, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.Hardswish()
        )
        # SE attention
        self.se = SEBlock(mid_ch)
        # Pointwise compress
        self.pw2 = nn.Sequential(
            nn.Conv2d(mid_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        # Residual connection (only when shapes match)
        self.use_residual = (stride == 1 and in_ch == out_ch)

    def forward(self, x):
        out = self.pw1(x)
        out = self.dw(out)
        out = self.se(out)
        out = self.pw2(out)
        if self.use_residual:
            out = out + x
        return out


class MotionCNN(nn.Module):
    """
    Deep flow feature extractor that mirrors the depth and
    structure of MobileNetV3-Small.

    Input : (B, 2, 224, 224)   Farneback flow in [-1,1]
    Output: (B, 64, 7, 7)      Rich motion features

    Architecture:
      Stem conv (2→16, stride 2)            224→112
      Block 1   (16→24, stride 2)           112→56
      Block 2   (24→32, stride 2)            56→28
      Block 3   (32→48, stride 1)            28→28
      Block 4   (48→64, stride 2)            28→14
      Block 5   (64→64, stride 1)            14→14
      AdaptiveAvgPool2d((7,7))               14→7

    The stem and first two blocks handle the rapid spatial
    reduction from 224 to 28. Blocks 3-5 add depth at the
    14x14 scale where motion patterns are most meaningful.
    SE attention at each block lets the network emphasise
    informative flow directions (convergent, divergent)
    over background noise.

    Why 64 output channels instead of 32?
    The spatial branch outputs 64 channels after spatial_reduce.
    Matching the flow branch capacity (also 64) gives equal
    weight to both streams in the fused 128-channel LSTM input.
    """

    def __init__(self):
        super().__init__()

        # Stem: initial conv to get off the 2-channel bottleneck
        self.stem = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.Hardswish()
        )   # (B, 16, 112, 112)

        # Blocks with progressive channel expansion and spatial reduction
        self.blocks = nn.Sequential(
            MotionResBlock(16,  24, stride=2),   # →(B,24,56,56)
            MotionResBlock(24,  32, stride=2),   # →(B,32,28,28)
            MotionResBlock(32,  48, stride=1),   # →(B,48,28,28)  depth
            MotionResBlock(48,  64, stride=2),   # →(B,64,14,14)
            MotionResBlock(64,  64, stride=1),   # →(B,64,14,14)  depth
        )

        # Final projection + pool to 7x7
        self.final = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.Hardswish(),
            nn.AdaptiveAvgPool2d((7, 7))         # →(B,64,7,7)
        )

        # Dropout to prevent flow branch from dominating
        self.drop = nn.Dropout2d(0.15)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.final(x)
        x = self.drop(x)
        return x   # (B, 64, 7, 7)


# ============================================================
# CROWD NET
# ============================================================

class CrowdNet(nn.Module):

    def __init__(self):
        super().__init__()

        # ── Hidden dimension ──────────────────────────────────
        HIDDEN_DIM      = 128
        self.hidden_dim = HIDDEN_DIM

        # ── MobileNetV3-Small backbone ────────────────────────
        mobilenet = models.mobilenet_v3_small(weights=None)
        if os.path.exists("mobilenet_v3_small-047dcff4.pth"):
            print("LOADING LOCAL MOBILENET WEIGHTS")
            state_dict = torch.load(
                "mobilenet_v3_small-047dcff4.pth",
                weights_only=True
            )
            mobilenet.load_state_dict(state_dict)

        features = mobilenet.features
        self.backbone_stage1 = features[:3]   # →(B,24,28,28)
        self.backbone_stage2 = features[3:]   # →(B,576,7,7)

        # Freeze Stage 1 — low-level detectors transfer well
        for param in self.backbone_stage1.parameters():
            param.requires_grad = False

        # ── Density head (tapped at Stage 1, 28x28) ──────────
        # 24ch at 28x28 — high-resolution for accurate counting
        self.density_head = nn.Sequential(
            nn.Conv2d(24,  64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64,  64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64,  32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32,   1, kernel_size=1)
            # No activation — density clamped externally
        )

        # ── Spatial feature reduction: 576→64 ────────────────
        self.spatial_reduce = nn.Sequential(
            nn.Conv2d(576, 64, kernel_size=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(0.2)
        )

        # ── MotionCNN flow branch: 2→64 channels, 224→7 ──────
        # Replaces the shallow 3-layer stride-2 conv stack.
        # Now has 5 residual blocks with SE attention,
        # matching the depth of the MobileNet spatial branch.
        self.motion_cnn = MotionCNN()

        # ── ConvLSTM ──────────────────────────────────────────
        # input_dim = 64 (spatial) + 64 (flow) = 128
        # Increased from 96 because MotionCNN outputs 64ch
        # (was 32ch from the old flow_reduce)
        self.convlstm = ConvLSTMCell(
            input_dim=128,    # 64 spatial + 64 motion
            hidden_dim=HIDDEN_DIM
        )

        # ── Classification head ───────────────────────────────
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(HIDDEN_DIM, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 3)
        )

    def forward(self, frames, flows):
        """
        frames : (B, T, 1, 224, 224)
        flows  : (B, T, 2, 224, 224)
        """
        B, T, _, H, W = frames.shape
        h, c = None, None
        last_stage1 = None

        for t in range(T):

            frame = frames[:, t]            # (B, 1, H, W)
            flow  = flows[:, t]             # (B, 2, H, W)

            # Grayscale → 3-channel for MobileNet
            frame_rgb = frame.repeat(1, 3, 1, 1)

            # Spatial features
            stage1 = self.backbone_stage1(frame_rgb)   # (B,24,28,28)
            stage2 = self.backbone_stage2(stage1)      # (B,576,7,7)
            last_stage1 = stage1

            spatial_small = self.spatial_reduce(stage2)   # (B,64,7,7)

            # Motion features from MotionCNN
            flow_features = self.motion_cnn(flow)         # (B,64,7,7)

            # Fuse: 64+64 = 128 channels
            fused = torch.cat([spatial_small, flow_features], dim=1)

            # Init LSTM on first step
            if h is None:
                _, _, FH, FW = fused.shape
                h = torch.zeros(B, self.hidden_dim, FH, FW, device=fused.device)
                c = torch.zeros_like(h)

            h, c = self.convlstm(fused, (h, c))

        # Density from last frame's Stage 1 features (28x28)
        density_map = self.density_head(last_stage1)

        # Classification from final LSTM hidden state
        pooled = self.pool(h).view(B, -1)
        logits = self.classifier(pooled)

        return logits, density_map


# ── Instantiate ───────────────────────────────────────────────
model = CrowdNet().to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print("=" * 60)
print("MODEL INITIALIZED")
print(f"Trainable params : {trainable:,}")
print(f"Total params     : {total:,}")
print("=" * 60)

# Shape verification
with torch.no_grad():
    df = torch.zeros(1, 15, 1, 224, 224).to(device)
    dfl = torch.zeros(1, 15, 2, 224, 224).to(device)
    dl, dd = model(df, dfl)
    print(f"Logits shape   : {dl.shape}")    # (1, 3)
    print(f"Density shape  : {dd.shape}")    # (1, 1, 28, 28)
    print(f"ConvLSTM input : {128} channels (64 spatial + 64 motion)")
    print(f"MotionCNN      : {model.motion_cnn}")

LOADING LOCAL MOBILENET WEIGHTS
MODEL INITIALIZED
Trainable params : 2,314,488
Total params     : 2,319,560
Logits shape   : torch.Size([1, 3])
Density shape  : torch.Size([1, 1, 28, 28])
ConvLSTM input : 128 channels (64 spatial + 64 motion)
MotionCNN      : MotionCNN(
  (stem): Sequential(
    (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Hardswish()
  )
  (blocks): Sequential(
    (0): MotionResBlock(
      (pw1): Sequential(
        (0): Conv2d(16, 48, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (dw): Sequential(
        (0): Conv2d(48, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=48, bias=False)
        (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): 

In [ ]:
print(model.spatial_reduce)

In [4]:
#cell 4
# ============================================================
# CELL 4 : LOSSES + OPTIMIZER + SCHEDULER
# ============================================================

classification_criterion = nn.CrossEntropyLoss()

density_criterion = nn.SmoothL1Loss(beta=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=5e-4       # increased from 1e-4 to fight overfitting
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,             # reduced from 5 — tighter for 40 epochs
    min_lr=1e-6
)

print("=" * 60)
print("LOSSES + OPTIMIZER INITIALIZED")
print(f"LR           : {optimizer.param_groups[0]['lr']}")
print(f"Weight decay : {optimizer.param_groups[0]['weight_decay']}")
print("=" * 60)

LOSSES + OPTIMIZER INITIALIZED
LR           : 0.0003
Weight decay : 0.0005


In [6]:
# ============================================================
# CELL 5 : TRAINING + VALIDATION LOOPS
# ============================================================

from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, epoch, total_epochs):
    """
    Curriculum augmentation: aug_prob starts low and increases
    with epoch so the model first learns the clean signal,
    then progressively adapts to degraded inputs.

    Epoch 1-5  : aug_prob = 0.3  (light augmentation)
    Epoch 6-15 : aug_prob = 0.5  (moderate)
    Epoch 16+  : aug_prob = 0.7  (heavy — full robustness training)

    The aug_prob is set on the dataset directly so it takes
    effect on the next DataLoader iteration.
    """

    # Set augmentation probability based on epoch
    if epoch <= 5:
        aug_prob = 0.3
    elif epoch <= 15:
        aug_prob = 0.5
    else:
        aug_prob = 0.7

    # Update aug_prob on the underlying dataset
    loader.dataset.aug_prob = aug_prob

    model.train()
    total_loss = total_cls_loss = total_den_loss = 0.0
    correct = total = 0

    loop = tqdm(enumerate(loader), total=len(loader), leave=True)

    for batch_idx, (frames, flows, labels, density_targets) in loop:

        frames          = frames.to(device)
        flows           = flows.to(device)
        labels          = labels.to(device)
        density_targets = density_targets.to(device)

        optimizer.zero_grad()

        logits, density_preds = model(frames, flows)
        density_preds = torch.clamp(density_preds, min=0.0)

        cls_loss = classification_criterion(logits, labels)
        den_loss = density_criterion(density_preds, density_targets)
        loss     = cls_loss + 0.001 * den_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss     += loss.item()
        total_cls_loss += cls_loss.item()
        total_den_loss += den_loss.item()

        preds    = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        acc      = 100.0 * correct / total

        loop.set_description(
            f"TRAIN ep{epoch} aug={aug_prob:.1f} | "
            f"LOSS {total_loss/(batch_idx+1):.4f} | "
            f"CLS {total_cls_loss/(batch_idx+1):.4f} | "
            f"ACC {acc:.2f}%"
        )

    return (
        total_loss     / len(loader),
        total_cls_loss / len(loader),
        total_den_loss / len(loader),
        correct / total
    )


def validate_one_epoch(model, loader):

    model.eval()
    total_loss = total_cls_loss = total_den_loss = 0.0
    correct = total = 0

    with torch.no_grad():
        loop = tqdm(enumerate(loader), total=len(loader), leave=True)
        for batch_idx, (frames, flows, labels, density_targets) in loop:

            frames          = frames.to(device)
            flows           = flows.to(device)
            labels          = labels.to(device)
            density_targets = density_targets.to(device)

            logits, density_preds = model(frames, flows)
            density_preds = torch.clamp(density_preds, min=0.0)

            cls_loss = classification_criterion(logits, labels)
            den_loss = density_criterion(density_preds, density_targets)
            loss     = cls_loss + 0.001 * den_loss

            total_loss     += loss.item()
            total_cls_loss += cls_loss.item()
            total_den_loss += den_loss.item()

            preds    = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            acc      = 100.0 * correct / total

            loop.set_description(
                f"VAL | "
                f"LOSS {total_loss/(batch_idx+1):.4f} | "
                f"CLS {total_cls_loss/(batch_idx+1):.4f} | "
                f"ACC {acc:.2f}%"
            )

    return (
        total_loss     / len(loader),
        total_cls_loss / len(loader),
        total_den_loss / len(loader),
        correct / total
    )

In [7]:
# ============================================================
# CELL 6 : TRAINING DRIVER
# ============================================================

# ============================================================
# DATASETS
# ============================================================
from torch.utils.data import DataLoader

# In Cell 6, update dataset instantiation only.
# aug_prob=0.3 is the starting value; Cell 5 updates it per epoch.

train_dataset = CrowdDataset(
    "split_dataset_v7/train",
    seq_length=15,
    stride=4,
    training=True,
    aug_prob=0.3        # starting low — curriculum increases it
)
val_dataset = CrowdDataset(
    "split_dataset_v7/val",
    seq_length=15,
    stride=4,
    training=False      # no augmentation on val/test ever
)
test_dataset = CrowdDataset(
    "split_dataset_v7/test",
    seq_length=15,
    stride=4,
    training=False
)

# Also update the training loop call to pass epoch number:
# train_loss, train_cls, train_den, train_acc = train_one_epoch(
#     model, train_loader, optimizer, epoch+1, EPOCHS
# )

train_loader = DataLoader(
    train_dataset, batch_size=4, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=4, shuffle=False, num_workers=0
)

# ── Sanity checks before training ────────────────────────────
print(f"Train windows : {len(train_dataset)}")
print(f"Val windows   : {len(val_dataset)}")
print(f"Test windows  : {len(test_dataset)}")

sample_frames, sample_flows, sample_labels, sample_density = next(iter(train_loader))
print(f"\nFrames shape  : {sample_frames.shape}")
print(f"Flows shape   : {sample_flows.shape}")
print(f"Labels        : {sample_labels}")
print(f"Density shape : {sample_density.shape}")
print(f"\nFlow min  : {sample_flows.min():.4f}")
print(f"Flow max  : {sample_flows.max():.4f}")
print(f"Flow mean : {sample_flows.mean():.4f}")
print(f"Flow std  : {sample_flows.std():.4f}")

# Flow std should be > 0.05 after the divisor fix
# Label distribution should be mixed across all three classes


# Full label distribution — add this right after dataset instantiation
print("\n=== FULL LABEL DISTRIBUTION ===new")
for split_name, dataset in [
    ("train", train_dataset),
    ("val",   val_dataset),
    ("test",  test_dataset)
]:
    counts = {0: 0, 1: 0, 2: 0}
    for w in dataset.windows:
        counts[w["label"]] += 1   # key must match Cell 2
    total_w = sum(counts.values())
    print(f"{split_name:6s} | {counts} | total: {total_w}")

print("\n=== FLOW DISTRIBUTION CHECK ===")
for split_name, dataset in [
    ("train", train_dataset),
    ("val",   val_dataset),
    ("test",  test_dataset)
]:
    # Use a shuffled loader for this check only
    check_loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)
    stds, label_counts = [], {0: 0, 1: 0, 2: 0}
    for frames, flows, labels, density in check_loader:
        stds.append(flows.std().item())
        for l in labels.tolist():
            label_counts[l] += 1
        if len(stds) >= 20:  # 20 batches × 4 = 80 samples, enough to see all classes
            break
    print(
        f"{split_name:6s} | "
        f"flow std: {sum(stds)/len(stds):.4f} | "
        f"labels: {label_counts}"
    )

# ── Do not proceed if flow std < 0.05 ────────────────────────
train_std = sum(
    flows.std().item()
    for _, flows, _, _ in [next(iter(train_loader))]
)
assert train_std > 0.03, (
    f"Flow std {train_std:.4f} is too low. "
    f"Check Cell 2 flow normalization divisor."
)

print("\nSANITY CHECKS PASSED — starting training\n")

# ── Training loop ─────────────────────────────────────────────
EPOCHS = 50
best_val_loss = 1e9
patience = 10
patience_counter = 0

for epoch in range(EPOCHS):

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch+1}/{EPOCHS}")
    print("=" * 60)

    
    train_loss, train_cls, train_den, train_acc = train_one_epoch(model, train_loader, optimizer, epoch+1, EPOCHS)

    val_loss, val_cls, val_den, val_acc = validate_one_epoch(
        model, val_loader
    )

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"\nTRAIN  | loss {train_loss:.4f} | cls {train_cls:.4f} | den {train_den:.4f} | acc {train_acc*100:.2f}%")
    print(f"VAL    | loss {val_loss:.4f} | cls {val_cls:.4f} | den {val_den:.4f} | acc {val_acc*100:.2f}%")
    print(f"LR     : {current_lr:.6f}")

    # Replace the checkpoint block with this cleaner version
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_crowd_model.pth")
        print(f"MODEL IMPROVED → SAVED  (val_acc={val_acc*100:.2f}%)")
    else:
        patience_counter += 1
        print(f"NO IMPROVEMENT ({patience_counter}/{patience})")
   
    if patience_counter >= patience:
        print("\nEARLY STOPPING TRIGGERED")
        break
# After early stopping or loop end:
print(f"\nBest val loss : {best_val_loss:.4f}")
print(f"Best val acc  : {best_val_acc*100:.2f}%")
print("\nTRAINING COMPLETE")

LOADING DATASET : split_dataset_v7/train
seq_length      : 15
stride          : 4
training mode   : True
aug_prob        : 0.3
TOTAL WINDOWS : 1260
LOADING DATASET : split_dataset_v7/val
seq_length      : 15
stride          : 4
training mode   : False
aug_prob        : N/A (val/test)
TOTAL WINDOWS : 270
LOADING DATASET : split_dataset_v7/test
seq_length      : 15
stride          : 4
training mode   : False
aug_prob        : N/A (val/test)
TOTAL WINDOWS : 270
Train windows : 1260
Val windows   : 270
Test windows  : 270

Frames shape  : torch.Size([4, 15, 1, 224, 224])
Flows shape   : torch.Size([4, 15, 2, 224, 224])
Labels        : tensor([1, 0, 0, 1])
Density shape : torch.Size([4, 1, 28, 28])

Flow min  : -1.0000
Flow max  : 1.0000
Flow mean : 0.0063
Flow std  : 0.1108

=== FULL LABEL DISTRIBUTION ===new
train  | {0: 420, 1: 420, 2: 420} | total: 1260
val    | {0: 90, 1: 90, 2: 90} | total: 270
test   | {0: 90, 1: 90, 2: 90} | total: 270

=== FLOW DISTRIBUTION CHECK ===
train  | flow 

TRAIN ep1 aug=0.3 | LOSS 1.0612 | CLS 1.0611 | ACC 38.60%:  18%|███▉                  | 57/315 [02:22<11:13,  2.61s/it]


FLOW VERIFICATION (idx=0)
  flow std  : 0.1111
  density sum : 151.0 agents



TRAIN ep1 aug=0.3 | LOSS 0.7357 | CLS 0.7356 | ACC 64.76%: 100%|█████████████████████| 315/315 [12:48<00:00,  2.44s/it]
  0%|                                                                                           | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std  : 0.1000
  density sum : 138.0 agents



...


FLOW VERIFICATION (idx=0)
  flow std  : 0.0835
  density sum : 151.0 agents



TRAIN ep25 aug=0.7 | LOSS 0.0082 | CLS 0.0082 | ACC 99.84%: 100%|████████████████████| 315/315 [12:33<00:00,  2.39s/it]
  0%|                                                                                           | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std  : 0.1000
  density sum : 138.0 agents



VAL | LOSS 0.0727 | CLS 0.0726 | ACC 98.89%: 100%|█████████████████████████████████████| 68/68 [01:45<00:00,  1.55s/it]


TRAIN  | loss 0.0082 | cls 0.0082 | den 0.0064 | acc 99.84%
VAL    | loss 0.0727 | cls 0.0726 | den 0.0053 | acc 98.89%
LR     : 0.000075
NO IMPROVEMENT (10/10)

EARLY STOPPING TRIGGERED

Best val loss : 0.0452
Best val acc  : 98.52%

TRAINING COMPLETE


In [10]:
# ============================================================
# CELL 7 : TESTING + ROBUSTNESS
# ============================================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

# ============================================================
# CORRUPTION
# ============================================================

def corrupt_frames(frames, flows, mode="gaussian"):
    """
    Applies realistic sensor corruption to frames and flows.

    frames : (B, T, 1, H, W)   normalized [0, 1]
    flows  : (B, T, 2, H, W)   normalized [-1, 1]

    Frame noise sigma = 0.08 on [0,1] = ~20/255 pixel noise.
    This is a moderate real-world sensor noise level.

    Flow noise must be calibrated to the flow signal scale.
    Clean flow std ≈ 0.10. To avoid destroying the signal:
      - mild corruption   : flow sigma = 0.01  (10% of signal)
      - moderate          : flow sigma = 0.02  (20% of signal)
      - severe (stress)   : flow sigma = 0.05  (50% of signal)

    The previous value of 0.12 (120% of signal) was incorrect
    and completely drowned the flow signal.
    """

    frames = frames.clone()
    flows  = flows.clone()

    if mode == "gaussian":
        # Moderate frame noise — realistic thermal sensor noise
        frames = frames + torch.randn_like(frames) * 0.08

        # Mild flow corruption corresponding to what Farneback
        # actually produces when given mildly noisy frames.
        # 0.02 = ~20% of clean flow std (0.10), which is a
        # realistic estimate of Farneback noise transfer.
        flows  = flows  + torch.randn_like(flows)  * 0.02

    elif mode == "blur":
        # Spatial blur on frames
        B, T, C, H, W = frames.shape
        blurred = []
        for b in range(B):
            temp = []
            for t in range(T):
                x = frames[b, t].cpu().numpy()[0]
                x = cv2.GaussianBlur(x, (9, 9), 0)
                temp.append(torch.tensor(x).unsqueeze(0))
            blurred.append(torch.stack(temp))
        frames = torch.stack(blurred)

        # Blur smooths intensity gradients that Farneback
        # relies on — empirically reduces flow magnitude ~15%
        flows = flows * 0.85

    frames = torch.clamp(frames, 0.0, 1.0)
    flows  = torch.clamp(flows, -1.0, 1.0)

    return frames, flows


# ============================================================
# EVALUATE
# ============================================================

def evaluate_model(model, loader, corruption=None):

    model.eval()

    preds_all  = []
    labels_all = []
    mae_total  = 0.0
    n_batches  = 0

    with torch.no_grad():

        loop = tqdm(loader)

        for frames, flows, labels, density_targets in loop:

            if corruption is not None:
                frames, flows = corrupt_frames(frames, flows, corruption)

            frames          = frames.to(device)
            flows           = flows.to(device)
            labels          = labels.to(device)
            density_targets = density_targets.to(device)

            logits, density_preds = model(frames, flows)

            # Clamp before count computation — density head has
            # no output activation so produces negative values
            # which corrupt the sum-based count estimate
            density_preds = torch.clamp(density_preds, min=0.0)

            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

            pred_counts = density_preds.sum(dim=[1,2,3]).cpu().numpy()
            true_counts = density_targets.sum(dim=[1,2,3]).cpu().numpy()
            mae_total  += np.mean(np.abs(pred_counts - true_counts))
            n_batches  += 1

    acc = accuracy_score(labels_all, preds_all)

    print("\n" + "=" * 60)
    print("CLEAN EVALUATION" if corruption is None
          else f"CORRUPTION : {corruption}")
    print("=" * 60)
    print(f"\nACCURACY  : {acc*100:.2f}%")
    print(f"COUNT MAE : {mae_total/n_batches:.2f} agents")
    print("\nCLASSIFICATION REPORT\n")
    print(classification_report(
        labels_all, preds_all,
        target_names=["Normal", "Bottleneck", "Panic"],
        digits=4, zero_division=0
    ))
    print("\nCONFUSION MATRIX\n")
    print(confusion_matrix(labels_all, preds_all))


# ============================================================
# LOAD AND EVALUATE
# ============================================================

model.load_state_dict(
    torch.load("best_crowd_model.pth", weights_only=True)
)
print("BEST MODEL LOADED")

evaluate_model(model, test_loader, corruption=None)
evaluate_model(model, test_loader, corruption="gaussian")
evaluate_model(model, test_loader, corruption="blur")

# ============================================================
# GAUSSIAN SEVERITY SWEEP
# ============================================================
# Tests model at increasing flow noise levels to find the
# break point. This tells you the operational noise budget.

print("\n" + "=" * 60)
print("GAUSSIAN SEVERITY SWEEP")
print("=" * 60)
print(f"Clean flow std : ~0.10")
print(f"{'Flow sigma':>12}  {'Noise/signal':>13}  {'Accuracy':>10}  {'MAE':>8}")
print("-" * 50)

for flow_sigma in [0.0, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15]:

    model.eval()
    preds_all  = []
    labels_all = []
    mae_total  = 0.0
    n_batches  = 0

    with torch.no_grad():
        for frames, flows, labels, density_targets in test_loader:

            frames_c = frames + torch.randn_like(frames) * 0.08
            frames_c = torch.clamp(frames_c, 0.0, 1.0)

            flows_c  = flows + torch.randn_like(flows) * flow_sigma
            flows_c  = torch.clamp(flows_c, -1.0, 1.0)

            frames_c = frames_c.to(device)
            flows_c  = flows_c.to(device)
            labels   = labels.to(device)
            density_targets = density_targets.to(device)

            logits, density_preds = model(frames_c, flows_c)
            density_preds = torch.clamp(density_preds, min=0.0)

            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

            pred_counts = density_preds.sum(dim=[1,2,3]).cpu().numpy()
            true_counts = density_targets.sum(dim=[1,2,3]).cpu().numpy()
            mae_total  += np.mean(np.abs(pred_counts - true_counts))
            n_batches  += 1

    acc          = 100 * accuracy_score(labels_all, preds_all)
    avg_mae      = mae_total / n_batches
    noise_ratio  = flow_sigma / 0.10   # relative to clean flow std

    print(f"{flow_sigma:>12.3f}  {noise_ratio:>13.2f}x  {acc:>9.2f}%  {avg_mae:>7.2f}")

BEST MODEL LOADED


  0%|                                                                                           | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents



100%|██████████████████████████████████████████████████████████████████████████████████| 68/68 [01:45<00:00,  1.56s/it]



CLEAN EVALUATION

ACCURACY  : 97.78%
COUNT MAE : 1.75 agents

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      Normal     0.9884    0.9444    0.9659        90
  Bottleneck     0.9468    0.9889    0.9674        90
       Panic     1.0000    1.0000    1.0000        90

    accuracy                         0.9778       270
   macro avg     0.9784    0.9778    0.9778       270
weighted avg     0.9784    0.9778    0.9778       270


CONFUSION MATRIX

[[85  5  0]
 [ 1 89  0]
 [ 0  0 90]]


...


CORRUPTION : gaussian

ACCURACY  : 96.67%
COUNT MAE : 6.79 agents

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      Normal     0.9175    0.9889    0.9519        90
  Bottleneck     0.9880    0.9111    0.9480        90
       Panic     1.0000    1.0000    1.0000        90

    accuracy                         0.9667       270
   macro avg     0.9685    0.9667    0.9666       270
weighted avg     0.9685    0.9667    0.9666       270


CONFUSION MATRIX

[[89  1  0]
 [ 8 82  0]
 [ 0  0 90]]


  0%|                                                                                           | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents



100%|██████████████████████████████████████████████████████████████████████████████████| 68/68 [01:47<00:00,  1.58s/it]



CORRUPTION : blur

ACCURACY  : 97.04%
COUNT MAE : 1.80 agents

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      Normal     0.9881    0.9222    0.9540        90
  Bottleneck     0.9271    0.9889    0.9570        90
       Panic     1.0000    1.0000    1.0000        90

    accuracy                         0.9704       270
   macro avg     0.9717    0.9704    0.9703       270
weighted avg     0.9717    0.9704    0.9703       270


CONFUSION MATRIX

[[83  7  0]
 [ 1 89  0]
 [ 0  0 90]]

GAUSSIAN SEVERITY SWEEP
Clean flow std : ~0.10
  Flow sigma   Noise/signal    Accuracy       MAE
--------------------------------------------------

FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents

       0.000           0.00x      96.30%     6.81

FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents

       0.010           0.10x      97.04%     6.86

FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 a

In [11]:
# ============================================================
# CELL 8 : TEMPORAL SANITY TESTS
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Tests whether:
#
#   - ConvLSTM is truly using temporal order
#   - Optical flow branch contributes meaningfully
#   - CNN spatial shortcut dominance still exists
#
# ============================================================

from sklearn.metrics import accuracy_score

# ============================================================
# TEMPORAL SHUFFLE
# ============================================================

def temporal_shuffle(frames, flows):

    B, T, C, H, W = frames.shape

    shuffled_frames = []

    shuffled_flows = []

    for b in range(B):

        perm = torch.randperm(T)

        shuffled_frames.append(
            frames[b, perm]
        )

        shuffled_flows.append(
            flows[b, perm]
        )

    return (
        torch.stack(shuffled_frames),
        torch.stack(shuffled_flows)
    )

# ============================================================
# SINGLE FRAME DUPLICATION
# ============================================================

def duplicate_single_frame(frames, flows):

    B, T, C, H, W = frames.shape

    duplicated_frames = []

    duplicated_flows = []

    for b in range(B):

        # choose middle frame
        idx = T // 2

        frame = frames[b, idx]

        repeated_frames = frame.unsqueeze(0).repeat(
            T,
            1,
            1,
            1
        )

        # ----------------------------------------------------
        # FLOWS BECOME ZERO
        # ----------------------------------------------------

        repeated_flows = torch.zeros_like(
            flows[b]
        )

        duplicated_frames.append(
            repeated_frames
        )

        duplicated_flows.append(
            repeated_flows
        )

    return (
        torch.stack(duplicated_frames),
        torch.stack(duplicated_flows)
    )

# ============================================================
# ZERO FLOW
# ============================================================

def zero_flow(frames, flows):

    return (
        frames,
        torch.zeros_like(flows)
    )

# ============================================================
# RANDOM FLOW
# ============================================================

def random_flow(frames, flows):

    random_flows = torch.randn_like(flows)

    random_flows = torch.clamp(
        random_flows,
        -1.0,
        1.0
    )

    return (
        frames,
        random_flows
    )

# ============================================================
# REVERSE TEMPORAL ORDER
# ============================================================

def reverse_temporal(frames, flows):

    reversed_frames = torch.flip(
        frames,
        dims=[1]
    )

    reversed_flows = torch.flip(
        flows,
        dims=[1]
    )

    return (
        reversed_frames,
        reversed_flows
    )

# ============================================================
# EVALUATION CORE
# ============================================================

def evaluate_sanity_mode(
    model,
    loader,
    mode_name,
    transform_fn=None
):

    model.eval()

    all_preds = []

    all_labels = []

    with torch.no_grad():

        loop = tqdm(
            loader,
            desc=f"TEST : {mode_name}"
        )

        for (
            frames,
            flows,
            labels,
            density_targets
        ) in loop:

            # ------------------------------------------------
            # APPLY TRANSFORMATION
            # ------------------------------------------------

            if transform_fn is not None:

                frames, flows = transform_fn(
                    frames,
                    flows
                )

            frames = frames.to(device)

            flows = flows.to(device)

            labels = labels.to(device)

            logits, _ = model(
                frames,
                flows
            )

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    print("\n" + "=" * 60)
    print(f"SANITY TEST : {mode_name}")
    print("=" * 60)

    print(f"\nAccuracy : {acc*100:.2f}%")

    return acc

# ============================================================
# MASTER SANITY SUITE
# ============================================================

def run_temporal_sanity_suite(
    model,
    test_loader
):

    print("\n")
    print("=" * 70)
    print("TEMPORAL SANITY SUITE INITIALIZED")
    print("=" * 70)

    results = {}

    # ========================================================
    # NORMAL
    # ========================================================

    results["normal"] = evaluate_sanity_mode(
        model,
        test_loader,
        "NORMAL",
        None
    )

    # ========================================================
    # SHUFFLED TIME
    # ========================================================

    results["shuffled"] = evaluate_sanity_mode(
        model,
        test_loader,
        "TEMPORAL SHUFFLE",
        temporal_shuffle
    )

    # ========================================================
    # REVERSED TIME
    # ========================================================

    results["reversed"] = evaluate_sanity_mode(
        model,
        test_loader,
        "TEMPORAL REVERSE",
        reverse_temporal
    )

    # ========================================================
    # DUPLICATED FRAME
    # ========================================================

    results["duplicated"] = evaluate_sanity_mode(
        model,
        test_loader,
        "SINGLE FRAME DUPLICATION",
        duplicate_single_frame
    )

    # ========================================================
    # ZERO FLOW
    # ========================================================

    results["zero_flow"] = evaluate_sanity_mode(
        model,
        test_loader,
        "ZERO FLOW",
        zero_flow
    )

    # ========================================================
    # RANDOM FLOW
    # ========================================================

    results["random_flow"] = evaluate_sanity_mode(
        model,
        test_loader,
        "RANDOM FLOW",
        random_flow
    )

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print("\n")
    print("=" * 70)
    print("TEMPORAL SANITY SUMMARY")
    print("=" * 70)

    for k, v in results.items():

        print(f"{k:25s} : {v*100:.2f}%")

    print("\n")

    # ========================================================
    # INTERPRETATION
    # ========================================================

    normal_acc = results["normal"]

    shuffled_acc = results["shuffled"]

    duplicated_acc = results["duplicated"]

    zero_flow_acc = results["zero_flow"]

    print("=" * 70)
    print("INTERPRETATION")
    print("=" * 70)

    # --------------------------------------------------------
    # TEMPORAL DEPENDENCY
    # --------------------------------------------------------

    temporal_drop = normal_acc - shuffled_acc

    print(
        f"\nTemporal Shuffle Drop : "
        f"{temporal_drop*100:.2f}%"
    )

    if temporal_drop < 0.10:

        print(
            "WARNING : weak temporal dependency."
        )

    else:

        print(
            "GOOD : ConvLSTM uses temporal order."
        )

    # --------------------------------------------------------
    # SPATIAL SHORTCUT
    # --------------------------------------------------------

    duplication_drop = normal_acc - duplicated_acc

    print(
        f"\nSingle Frame Drop : "
        f"{duplication_drop*100:.2f}%"
    )

    if duplication_drop < 0.10:

        print(
            "WARNING : CNN spatial shortcut dominance."
        )

    else:

        print(
            "GOOD : temporal evolution matters."
        )

    # --------------------------------------------------------
    # FLOW CONTRIBUTION
    # --------------------------------------------------------

    flow_drop = normal_acc - zero_flow_acc

    print(
        f"\nZero Flow Drop : "
        f"{flow_drop*100:.2f}%"
    )

    if flow_drop < 0.05:

        print(
            "WARNING : optical flow branch contributes little."
        )

    else:

        print(
            "GOOD : motion branch contributes."
        )

    print("\n")
    print("=" * 70)
    print("SANITY SUITE COMPLETE")
    print("=" * 70)

# ============================================================
# RUN SANITY TESTS
# ============================================================

run_temporal_sanity_suite(
    model,
    test_loader
)



TEMPORAL SANITY SUITE INITIALIZED


TEST : NORMAL:   0%|                                                                            | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents



TEST : NORMAL: 100%|███████████████████████████████████████████████████████████████████| 68/68 [01:44<00:00,  1.54s/it]



SANITY TEST : NORMAL

Accuracy : 97.78%


...


SANITY TEST : ZERO FLOW

Accuracy : 41.85%


TEST : RANDOM FLOW:   0%|                                                                       | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents



TEST : RANDOM FLOW: 100%|██████████████████████████████████████████████████████████████| 68/68 [01:51<00:00,  1.63s/it]


SANITY TEST : RANDOM FLOW

Accuracy : 33.33%


TEMPORAL SANITY SUMMARY
normal                    : 97.78%
shuffled                  : 86.67%
reversed                  : 55.56%
duplicated                : 41.85%
zero_flow                 : 41.85%
random_flow               : 33.33%


INTERPRETATION

Temporal Shuffle Drop : 11.11%
GOOD : ConvLSTM uses temporal order.

Single Frame Drop : 55.93%
GOOD : temporal evolution matters.

Zero Flow Drop : 55.93%
GOOD : motion branch contributes.


SANITY SUITE COMPLETE


In [12]:
# ============================================================
# Revised CELL 9 : COMPREHENSIVE NOISE SWEEP + VISUALISATION
# ============================================================
#
# Tests each attack type at 6-8 severity levels to produce
# accuracy vs severity curves for the paper.
#
# Attacks covered:
#   1. Gaussian noise (frame sigma)
#   2. Salt and pepper (pixel probability)
#   3. Fog (contrast attenuation alpha)
#   4. Frame dropout (frame loss probability)
#   5. Motion blur (kernel size)
#   6. Brightness (multiplicative factor)
#   7. Combined fog + gaussian
#
# ============================================================

import torch
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from torch.utils.data import DataLoader

# ── Ensure we use the correct stride-5 test loader ────────────
test_dataset_eval = CrowdDataset(
    "split_dataset_v7/test",
    seq_length=15,
    stride=4,
    training=False
)
test_loader_eval = DataLoader(
    test_dataset_eval,
    batch_size=4,
    shuffle=False,
    num_workers=0
)
print(f"Test windows for sweep : {len(test_dataset_eval)}")

# ── Load best model ───────────────────────────────────────────
model.load_state_dict(
    torch.load("best_crowd_model.pth", weights_only=True)
)
model.eval()
print("Best model loaded for sweep.")


# ============================================================
# CORE ATTACK FUNCTION
# ============================================================

def apply_sweep_attack(frames, flows, attack_type, severity):
    """
    Applies a single attack at a given severity level.
    All inputs/outputs normalised: frames [0,1], flows [-1,1].

    severity meaning per attack:
      gaussian    : frame sigma (0.0 to 0.20)
      salt_pepper : pixel probability (0.0 to 0.10)
      fog         : alpha, contrast pull toward 0.5 (0.0 to 0.80)
      frame_drop  : probability of zeroing a frame (0.0 to 0.60)
      blur        : gaussian kernel size (1 to 19, odd only)
      brightness  : multiplicative factor (0.2 to 2.0)
      fog_gauss   : combined alpha=severity, sigma=severity*0.15
    """
    frames = frames.clone().float()
    flows  = flows.clone().float()

    if attack_type == "gaussian":
        sigma  = severity
        frames = frames + torch.randn_like(frames) * sigma
        # Farneback noise transfer: empirically ~20% of frame sigma
        flows  = flows  + torch.randn_like(flows)  * (sigma * 0.20)

    elif attack_type == "salt_pepper":
        prob = severity
        mask = torch.rand_like(frames)
        frames[mask < prob / 2]                    = 0.0
        frames[(mask >= prob/2) & (mask < prob)]   = 1.0
        # Flow spike suppression via median filter
        flows_np = flows.cpu().numpy()
        B, T, C, H, W = flows_np.shape
        for b in range(B):
            for t in range(T):
                flows_np[b, t, 0] = cv2.medianBlur(
                    flows_np[b, t, 0].astype(np.float32), 3
                )
                flows_np[b, t, 1] = cv2.medianBlur(
                    flows_np[b, t, 1].astype(np.float32), 3
                )
        flows = torch.from_numpy(flows_np)
        # Residual flow spikes
        fmask = torch.rand_like(flows)
        flows[fmask < prob / 8]                        = -1.0
        flows[(fmask >= prob/8) & (fmask < prob/4)]    =  1.0

    elif attack_type == "fog":
        alpha  = severity
        frames = (1 - alpha) * frames + alpha * 0.5
        flows  = flows * (1 - alpha * 0.6)

    elif attack_type == "frame_drop":
        prob = severity
        B, T, C, H, W = frames.shape
        for b in range(B):
            for t in range(T):
                if torch.rand(1).item() < prob:
                    frames[b, t] = 0.0
                    flows[b, t]  = 0.0

    elif attack_type == "blur":
        k = int(severity)
        if k % 2 == 0:
            k += 1   # kernel must be odd
        if k < 1:
            k = 1
        if k > 1:
            B, T, C, H, W = frames.shape
            blurred = []
            for b in range(B):
                temp = []
                for t in range(T):
                    x = frames[b, t].cpu().numpy()[0]
                    x = cv2.GaussianBlur(x, (k, k), 0)
                    temp.append(torch.tensor(x).unsqueeze(0))
                blurred.append(torch.stack(temp))
            frames = torch.stack(blurred)
        # Flow attenuation increases with blur strength
        attenuation = 1.0 - (k - 1) / 40.0
        flows = flows * max(attenuation, 0.3)

    elif attack_type == "brightness":
        factor = severity
        frames = frames * factor
        flows  = flows * (0.85 + factor * 0.10)

    elif attack_type == "fog_gauss":
        alpha = severity
        sigma = severity * 0.15
        frames = (1 - alpha) * frames + alpha * 0.5
        frames = frames + torch.randn_like(frames) * sigma
        flows  = flows * (1 - alpha * 0.5) + torch.randn_like(flows) * (sigma * 0.15)

    frames = torch.clamp(frames, 0.0, 1.0)
    flows  = torch.clamp(flows, -1.0, 1.0)
    return frames, flows


# ============================================================
# SINGLE SEVERITY EVALUATION
# ============================================================

def evaluate_at_severity(model, loader, attack_type, severity):
    """Returns (overall_acc, normal_acc, bottleneck_acc, panic_acc)."""
    model.eval()
    preds_all  = []
    labels_all = []

    with torch.no_grad():
        for frames, flows, labels, _ in loader:
            frames, flows = apply_sweep_attack(
                frames, flows, attack_type, severity
            )
            frames = frames.to(device)
            flows  = flows.to(device)
            labels = labels.to(device)

            logits, _ = model(frames, flows)
            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    preds_np  = np.array(preds_all)
    labels_np = np.array(labels_all)
    overall   = 100 * accuracy_score(labels_np, preds_np)

    # Per-class accuracy
    per_class = []
    for c in range(3):
        mask = labels_np == c
        if mask.sum() > 0:
            per_class.append(
                100 * (preds_np[mask] == labels_np[mask]).mean()
            )
        else:
            per_class.append(0.0)

    return overall, per_class[0], per_class[1], per_class[2]


# ============================================================
# SWEEP DEFINITIONS
# ============================================================

attacks = {
    "Gaussian Noise\n(frame sigma)": {
        "type":       "gaussian",
        "severities": [0.00, 0.02, 0.04, 0.06, 0.08,
                       0.10, 0.12, 0.15, 0.18, 0.20],
        "x_label":    "Frame sigma",
        "x_format":   "{:.2f}",
    },
    "Salt & Pepper\n(pixel probability)": {
        "type":       "salt_pepper",
        "severities": [0.000, 0.005, 0.010, 0.020,
                       0.030, 0.050, 0.070, 0.100],
        "x_label":    "Pixel probability",
        "x_format":   "{:.3f}",
    },
    "Fog / Haze\n(contrast alpha)": {
        "type":       "fog",
        "severities": [0.00, 0.10, 0.20, 0.30,
                       0.40, 0.50, 0.60, 0.70, 0.80],
        "x_label":    "Alpha (0=clear, 1=total grey)",
        "x_format":   "{:.2f}",
    },
    "Frame Dropout\n(loss probability)": {
        "type":       "frame_drop",
        "severities": [0.00, 0.10, 0.20, 0.30,
                       0.40, 0.50, 0.60],
        "x_label":    "Frame loss probability",
        "x_format":   "{:.2f}",
    },
    "Motion Blur\n(kernel size)": {
        "type":       "blur",
        "severities": [1, 3, 5, 7, 9, 11, 13, 15, 17, 19],
        "x_label":    "Gaussian kernel size (px)",
        "x_format":   "{:.0f}",
    },
    "Brightness Change\n(multiplicative factor)": {
        "type":       "brightness",
        "severities": [0.25, 0.40, 0.55, 0.70, 0.85,
                       1.00, 1.20, 1.40, 1.60, 1.80],
        "x_label":    "Scale factor (1.0 = clean)",
        "x_format":   "{:.2f}",
    },
    "Combined Fog+Gaussian\n(joint severity)": {
        "type":       "fog_gauss",
        "severities": [0.00, 0.10, 0.20, 0.30,
                       0.40, 0.50, 0.60, 0.70],
        "x_label":    "Joint severity (alpha; sigma=alpha×0.15)",
        "x_format":   "{:.2f}",
    },
}

CLASS_NAMES  = ["Normal", "Bottleneck", "Panic"]
CLASS_COLORS = ["#2196F3", "#FF9800", "#F44336"]


# ============================================================
# RUN ALL SWEEPS
# ============================================================

print("\n" + "="*60)
print("RUNNING COMPREHENSIVE NOISE SWEEPS")
print("This will take approximately 15-25 minutes.")
print("="*60)

results = {}   # attack_name → dict of lists

for attack_name, cfg in attacks.items():
    print(f"\n--- {attack_name.replace(chr(10), ' ')} ---")
    overall_list = []
    normal_list  = []
    bn_list      = []
    panic_list   = []

    for sev in cfg["severities"]:
        label = cfg["x_format"].format(sev)
        overall, n_acc, bn_acc, p_acc = evaluate_at_severity(
            model, test_loader_eval, cfg["type"], sev
        )
        overall_list.append(overall)
        normal_list.append(n_acc)
        bn_list.append(bn_acc)
        panic_list.append(p_acc)
        print(f"  severity={label:>8}  overall={overall:>6.2f}%  "
              f"N={n_acc:>6.2f}%  B={bn_acc:>6.2f}%  P={p_acc:>6.2f}%")

    results[attack_name] = {
        "severities": cfg["severities"],
        "x_label":    cfg["x_label"],
        "x_format":   cfg["x_format"],

        "overall":    overall_list,
        "Normal":     normal_list,
        "Bottleneck": bn_list,
        "Panic":      panic_list,
    }
    
print("\nAll sweeps complete.")



LOADING DATASET : split_dataset_v7/test
seq_length      : 15
stride          : 4
training mode   : False
aug_prob        : N/A (val/test)
TOTAL WINDOWS : 270
Test windows for sweep : 270
Best model loaded for sweep.

RUNNING COMPREHENSIVE NOISE SWEEPS
This will take approximately 15-25 minutes.

--- Gaussian Noise (frame sigma) ---

FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents

  severity=    0.00  overall= 97.78%  N= 94.44%  B= 98.89%  P=100.00%

FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents

  severity=    0.02  overall= 98.15%  N= 96.67%  B= 97.78%  P=100.00%

FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents

  severity=    0.04  overall= 97.04%  N= 96.67%  B= 94.44%  P=100.00%

FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents

  severity=    0.06  overall= 97.04%  N= 96.67%  B= 94.44%  P=100.00%

FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agen

In [13]:
# ============================================================
# PATCH EXISTING RESULTS DICTIONARY
# ============================================================

for attack_name, cfg in attacks.items():

    if attack_name in results:

        results[attack_name]["x_format"] = cfg["x_format"]

print("results dictionary patched successfully.")

results dictionary patched successfully.


In [14]:
#Revised cell 9 plots
# ============================================================
# PLOT — 7-panel accuracy vs severity figure
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(22, 11))
fig.patch.set_facecolor("#0d1117")
axes_flat = axes.flatten()

# Hide the 8th panel (2x4 = 8, we have 7 attacks)
axes_flat[7].set_visible(False)

OVERALL_COLOR = "#FFFFFF"
LINE_STYLES   = ["-", "--", "-."]
ALPHA_FILL    = 0.12

for ax_idx, (attack_name, data) in enumerate(results.items()):
    ax = axes_flat[ax_idx]
    ax.set_facecolor("#161b22")

    sevs     = data["severities"]
    overall  = data["overall"]
    normal   = data["Normal"]
    bn       = data["Bottleneck"]
    panic    = data["Panic"]

    # Overall accuracy — thick white line
    ax.plot(sevs, overall, color=OVERALL_COLOR, linewidth=2.5,
            marker="o", markersize=5, label="Overall", zorder=5)

    # Per-class accuracy lines
    for cls_name, cls_data, col, ls in zip(
        CLASS_NAMES,
        [normal, bn, panic],
        CLASS_COLORS,
        LINE_STYLES
    ):
        ax.plot(sevs, cls_data, color=col, linewidth=1.4,
                linestyle=ls, marker="s", markersize=3.5,
                alpha=0.85, label=cls_name)
        # Shaded region between class line and overall
        ax.fill_between(sevs, cls_data, overall,
                        color=col, alpha=ALPHA_FILL)

    # Reference lines
    ax.axhline(y=95, color="#444444", linestyle=":", linewidth=0.8)
    ax.axhline(y=80, color="#333333", linestyle=":", linewidth=0.8)
    ax.axhline(y=33.33, color="#222222", linestyle="--",
               linewidth=0.7, label="Chance (33%)")

    # Shade the operational zone (>90%)
    ax.axhspan(90, 100, alpha=0.04, color="#4CAF50")

    # Annotate clean baseline
    clean_acc = overall[0]
    ax.annotate(
        f"Clean\n{clean_acc:.1f}%",
        xy=(sevs[0], clean_acc),
        xytext=(sevs[0], clean_acc - 12),
        fontsize=7, color="#AAAAAA",
        arrowprops=dict(arrowstyle="->", color="#555555", lw=0.8),
        ha="center"
    )

    # Find the point where overall drops below 90%
    cliff_sev = None
    for i in range(1, len(overall)):
        if overall[i] < 90.0:
            cliff_sev = sevs[i]
            ax.axvline(x=cliff_sev, color="#FF5722",
                       linestyle="--", linewidth=1.0, alpha=0.7)
            ax.text(cliff_sev, 36,
                    f"<90%\n@{data['x_format'].format(cliff_sev)}",
                    color="#FF5722", fontsize=6.5, ha="center",
                    va="bottom")
            break

    ax.set_xlim(min(sevs), max(sevs))
    ax.set_ylim(25, 105)
    ax.set_xlabel(data["x_label"], color="#CCCCCC", fontsize=8)
    ax.set_ylabel("Accuracy (%)", color="#CCCCCC", fontsize=8)
    ax.set_title(attack_name, color="white", fontsize=10,
                 fontweight="bold", pad=8)
    ax.tick_params(colors="#AAAAAA", labelsize=7)
    ax.spines["bottom"].set_color("#444444")
    ax.spines["left"].set_color("#444444")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#222222", linewidth=0.5)

# Shared legend (bottom of figure)
legend_elements = [
    plt.Line2D([0], [0], color=OVERALL_COLOR, linewidth=2.5,
               marker="o", markersize=5, label="Overall"),
    plt.Line2D([0], [0], color=CLASS_COLORS[0], linewidth=1.4,
               linestyle="-", marker="s", markersize=3.5, label="Normal"),
    plt.Line2D([0], [0], color=CLASS_COLORS[1], linewidth=1.4,
               linestyle="--", marker="s", markersize=3.5, label="Bottleneck"),
    plt.Line2D([0], [0], color=CLASS_COLORS[2], linewidth=1.4,
               linestyle="-.", marker="s", markersize=3.5, label="Panic"),
    plt.Line2D([0], [0], color="#FF5722", linewidth=1.0,
               linestyle="--", label="90% threshold crossing"),
    mpatches.Patch(facecolor="#4CAF50", alpha=0.15,
                   label="Operational zone (>90%)"),
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=6,
    fontsize=9,
    frameon=True,
    facecolor="#1c2128",
    edgecolor="#444444",
    labelcolor="white",
    bbox_to_anchor=(0.5, -0.02)
)

fig.suptitle(
    "CrowdNet Robustness — Accuracy vs Noise Attack Severity\n"
    "V7 Generator · MotionCNN · ConvLSTM(128) · seq_len=15 · stride=5",
    color="white", fontsize=13, fontweight="bold", y=1.01
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(
    "crowdnet_robustness_sweep.png",
    dpi=180,
    bbox_inches="tight",
    facecolor="#0d1117"
)
plt.close()
print("Saved: crowdnet_robustness_sweep.png")


# ============================================================
# SUMMARY TABLE — operational thresholds
# ============================================================

print("\n" + "="*70)
print("OPERATIONAL NOISE BUDGET SUMMARY")
print("(severity at which overall accuracy first drops below 90%)")
print("="*70)
print(f"{'Attack':<40}  {'Threshold':>12}  {'Clean acc':>10}")
print("-"*70)

for attack_name, data in results.items():
    clean_acc  = data["overall"][0]
    threshold  = "Never <90%"
    for i in range(1, len(data["overall"])):
        if data["overall"][i] < 90.0:
            sev_label = attacks[attack_name]["x_format"].format(
                data["severities"][i]
            )
            unit      = attacks[attack_name]["x_label"].split("(")[0].strip()
            threshold = f"{sev_label} {unit}"
            break
    name_clean = attack_name.replace("\n", " ")
    print(f"{name_clean:<40}  {threshold:>12}  {clean_acc:>9.2f}%")

print("="*70)


# ============================================================
# SECOND PLOT — consolidated overview bar chart
# ============================================================

# For each attack, show accuracy at 3 representative severities:
# mild (25th percentile), moderate (50th), severe (75th)

fig2, ax2 = plt.subplots(figsize=(16, 7))
fig2.patch.set_facecolor("#0d1117")
ax2.set_facecolor("#161b22")

attack_names_short = [
    "Gaussian", "Salt+Pepper", "Fog",
    "Frame\nDropout", "Blur", "Brightness", "Fog+Gauss"
]

x         = np.arange(len(attacks))
bar_width  = 0.22
severity_labels = ["Mild", "Moderate", "Severe"]
severity_colors = ["#4CAF50", "#FF9800", "#F44336"]

for s_idx, (s_label, s_color) in enumerate(
    zip(severity_labels, severity_colors)
):
    accs = []
    for attack_name, data in results.items():
        n    = len(data["severities"])
        idx  = [n // 4, n // 2, 3 * n // 4][s_idx]
        idx  = min(idx, n - 1)
        accs.append(data["overall"][idx])

    bars = ax2.bar(
        x + s_idx * bar_width,
        accs,
        bar_width,
        label=s_label,
        color=s_color,
        alpha=0.85,
        edgecolor="#000000",
        linewidth=0.5
    )

    for bar, acc in zip(bars, accs):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f"{acc:.0f}%",
            ha="center", va="bottom",
            color="white", fontsize=7, fontweight="bold"
        )

ax2.axhline(y=90, color="#FF5722", linestyle="--",
            linewidth=1.2, label="90% operational threshold")
ax2.axhline(y=33.33, color="#555555", linestyle=":",
            linewidth=0.8, label="Chance level (33.3%)")

ax2.set_xticks(x + bar_width)
ax2.set_xticklabels(attack_names_short, color="white", fontsize=9)
ax2.set_ylim(20, 110)
ax2.set_ylabel("Accuracy (%)", color="white", fontsize=10)
ax2.set_title(
    "CrowdNet Robustness Overview — Accuracy at Mild / Moderate / Severe Corruption",
    color="white", fontsize=12, fontweight="bold", pad=12
)
ax2.tick_params(colors="white", labelsize=9)
ax2.spines["bottom"].set_color("#444444")
ax2.spines["left"].set_color("#444444")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.grid(axis="y", color="#222222", linewidth=0.5)
ax2.legend(
    fontsize=9, frameon=True,
    facecolor="#1c2128", edgecolor="#444444",
    labelcolor="white", loc="lower right"
)

plt.tight_layout()
plt.savefig(
    "crowdnet_robustness_overview.png",
    dpi=180,
    bbox_inches="tight",
    facecolor="#0d1117"
)
plt.close()
print("Saved: crowdnet_robustness_overview.png")

Saved: crowdnet_robustness_sweep.png

OPERATIONAL NOISE BUDGET SUMMARY
(severity at which overall accuracy first drops below 90%)
Attack                                       Threshold   Clean acc
----------------------------------------------------------------------
Gaussian Noise (frame sigma)                Never <90%      97.78%
Salt & Pepper (pixel probability)         0.030 Pixel probability      97.78%
Fog / Haze (contrast alpha)                 0.80 Alpha      97.78%
Frame Dropout (loss probability)          0.40 Frame loss probability      97.78%
Motion Blur (kernel size)                 17 Gaussian kernel size      97.78%
Brightness Change (multiplicative factor)    Never <90%      80.74%
Combined Fog+Gaussian (joint severity)    0.70 Joint severity      97.78%
Saved: crowdnet_robustness_overview.png


In [15]:
# ============================================================
# CELL 9 : EXTENDED ROBUSTNESS ATTACKS
# ============================================================
#
# Tests six additional corruption modes:
#   1. Salt and pepper noise (sensor dead pixels)
#   2. Fog / contrast attenuation
#   3. Temporal frame dropout (packet loss)
#   4. Combined fog + gaussian
#   5. Heavy blur (severe motion blur)
#   6. Brightness fluctuation
#
# Each attack is applied to both frames and flows where
# physically appropriate.
# ============================================================

import torch
import numpy as np
import cv2
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm


test_dataset = CrowdDataset(
    "split_dataset_v7/test",
    seq_length=15,
    stride=4,
    training=False     # no augmentation
)
test_loader = DataLoader(
    test_dataset, batch_size=4, shuffle=False, num_workers=0
)



def apply_attack(frames, flows, mode, params):
    """
    Applies a named corruption attack to frames and flows.

    frames : (B, T, 1, H, W)  float [0,1]
    flows  : (B, T, 2, H, W)  float [-1,1]
    """
    frames = frames.clone().float()
    flows  = flows.clone().float()

    if mode == "salt_pepper":
        # Simulates dead/hot pixels — random pixels set to 0 or 1
        prob = params["prob"]
        mask = torch.rand_like(frames)
        frames[mask < prob / 2]                      = 0.0
        frames[(mask >= prob / 2) & (mask < prob)]   = 1.0
        # SP noise on frames creates spike-like flow artefacts
        flow_mask = torch.rand_like(flows)
        flows[flow_mask < prob / 4]                        = -1.0
        flows[(flow_mask >= prob / 4) & (flow_mask < prob / 2)] = 1.0

        # ADD: suppress flow spikes caused by S&P
        # Convert to numpy, apply median filter, convert back
        flows_np = flows.numpy()
        B, T, C, H, W = flows_np.shape
        for b in range(B):
            for t in range(T):
                flows_np[b, t, 0] = cv2.medianBlur(flows_np[b, t, 0], 3)
                flows_np[b, t, 1] = cv2.medianBlur(flows_np[b, t, 1], 3)
        flows = torch.from_numpy(flows_np)

        # Reduced flow spike injection since we're now suppressing
        flow_mask = torch.rand_like(flows)
        flows[flow_mask < prob / 8] = -1.0   # was prob/4
        flows[(flow_mask >= prob / 8) & (flow_mask < prob / 4)] = 1.0

    elif mode == "fog":
        # Simulates atmospheric haze — attenuates contrast toward 0.5
        alpha     = params["intensity"]
        fog_gray  = 0.5
        frames    = (1 - alpha) * frames + alpha * fog_gray
        # Fog reduces apparent motion contrast
        flows     = flows * (1 - alpha * 0.6)

    elif mode == "frame_dropout":
        # Simulates packet loss — random frames replaced with zeros
        prob = params["prob"]
        B, T, C, H, W = frames.shape
        for b in range(B):
            for t in range(T):
                if torch.rand(1).item() < prob:
                    frames[b, t] = 0.0
                    flows[b, t]  = 0.0

    elif mode == "fog_gaussian":
        # Combined fog + gaussian — realistic outdoor degradation
        alpha  = params["fog_intensity"]
        sigma  = params["gaussian_sigma"]
        frames = (1 - alpha) * frames + alpha * 0.5
        frames = frames + torch.randn_like(frames) * sigma
        flows  = flows * (1 - alpha * 0.5) + torch.randn_like(flows) * 0.015

    elif mode == "heavy_blur":
        # Severe motion blur — kernel 15x15
        k = params["kernel"]
        B, T, C, H, W = frames.shape
        blurred = []
        for b in range(B):
            temp = []
            for t in range(T):
                x = frames[b, t].cpu().numpy()[0]
                x = cv2.GaussianBlur(x, (k, k), 0)
                temp.append(torch.tensor(x).unsqueeze(0))
            blurred.append(torch.stack(temp))
        frames = torch.stack(blurred)
        # Heavy blur significantly attenuates flow
        flows = flows * 0.65

    elif mode == "brightness":
        # Simulates exposure variation — scale pixel values
        factor = params["factor"]
        frames = frames * factor
        # Brightness does not affect flow direction, only slightly magnitude
        flows  = flows * (0.9 + factor * 0.1)

    frames = torch.clamp(frames, 0.0, 1.0)
    flows  = torch.clamp(flows, -1.0, 1.0)
    return frames, flows


def evaluate_attack(model, loader, mode, params, label):
    """Runs a single attack and returns accuracy and MAE."""
    model.eval()
    preds_all  = []
    labels_all = []
    mae_total  = 0.0
    n_batches  = 0

    with torch.no_grad():
        for frames, flows, labels, density_targets in loader:
            frames, flows = apply_attack(frames, flows, mode, params)
            frames          = frames.to(device)
            flows           = flows.to(device)
            labels          = labels.to(device)
            density_targets = density_targets.to(device)

            logits, density_preds = model(frames, flows)
            density_preds = torch.clamp(density_preds, min=0.0)

            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

            pred_counts = density_preds.sum(dim=[1,2,3]).cpu().numpy()
            true_counts = density_targets.sum(dim=[1,2,3]).cpu().numpy()
            mae_total  += np.mean(np.abs(pred_counts - true_counts))
            n_batches  += 1

    acc = 100 * accuracy_score(labels_all, preds_all)
    mae = mae_total / n_batches

    print(f"\n{'='*60}")
    print(f"ATTACK : {label}")
    print(f"{'='*60}")
    print(f"Accuracy  : {acc:.2f}%")
    print(f"Count MAE : {mae:.2f} agents")
    print("\nClassification Report:")
    print(classification_report(
        labels_all, preds_all,
        target_names=["Normal", "Bottleneck", "Panic"],
        digits=4, zero_division=0
    ))
    print("Confusion Matrix:")
    print(confusion_matrix(labels_all, preds_all))
    return acc, mae


# ── Run all attacks ──────────────────────────────────────────
print("\n" + "="*60)
print("EXTENDED ROBUSTNESS SUITE")
print("="*60)

attacks = [
    ("salt_pepper",   {"prob": 0.02},                      "Salt & Pepper (2% dead pixels)"),
    ("salt_pepper",   {"prob": 0.05},                      "Salt & Pepper (5% dead pixels)"),
    ("fog",           {"intensity": 0.30},                  "Fog (30% contrast attenuation)"),
    ("fog",           {"intensity": 0.60},                  "Fog (60% contrast attenuation)"),
    ("frame_dropout", {"prob": 0.20},                      "Frame Dropout (20% packet loss)"),
    ("frame_dropout", {"prob": 0.40},                      "Frame Dropout (40% packet loss)"),
    ("fog_gaussian",  {"fog_intensity": 0.30,
                       "gaussian_sigma": 0.05},             "Combined Fog+Gaussian (outdoor)"),
    ("heavy_blur",    {"kernel": 15},                      "Heavy Blur (kernel 15x15)"),
    ("brightness",    {"factor": 0.5},                     "Brightness -50% (underexposed)"),
    ("brightness",    {"factor": 1.5},                     "Brightness +50% (overexposed)"),
]

summary = []
for mode, params, label in attacks:
    acc, mae = evaluate_attack(model, test_loader, mode, params, label)
    summary.append((label, acc, mae))

# ── Summary table ─────────────────────────────────────────────
print("\n" + "="*70)
print("EXTENDED ROBUSTNESS SUMMARY")
print("="*70)
print(f"{'Attack':<42}  {'Accuracy':>10}  {'Count MAE':>10}")
print("-"*70)
print(f"{'Clean baseline':<42}  {'97.50%':>10}  {'2.33':>10}")
for label, acc, mae in summary:
    print(f"{label:<42}  {acc:>9.2f}%  {mae:>9.2f}")
print("="*70)

LOADING DATASET : split_dataset_v7/test
seq_length      : 15
stride          : 4
training mode   : False
aug_prob        : N/A (val/test)
TOTAL WINDOWS : 270

EXTENDED ROBUSTNESS SUITE

FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents


ATTACK : Salt & Pepper (2% dead pixels)
Accuracy  : 92.59%
Count MAE : 2.44 agents

Classification Report:
              precision    recall  f1-score   support

      Normal     0.8500    0.9444    0.8947        90
  Bottleneck     0.9615    0.8333    0.8929        90
       Panic     0.9783    1.0000    0.9890        90

    accuracy                         0.9259       270
   macro avg     0.9299    0.9259    0.9255       270
weighted avg     0.9299    0.9259    0.9255       270

Confusion Matrix:
[[85  3  2]
 [15 75  0]
 [ 0  0 90]]

FLOW VERIFICATION (idx=0)
  flow std  : 0.1059
  density sum : 114.0 agents


ATTACK : Salt & Pepper (5% dead pixels)
Accuracy  : 74.07%
Count MAE : 10.69 agents

Classification Report:
      

In [17]:
# ============================================================
# CELL 10 : PER-SAMPLE VISUALISATION
# ============================================================
#
# Shows the model's prediction on individual test samples.
# For each sample:
#   - Grayscale frame (last frame in window)
#   - Optical flow visualised as HSV colour wheel
#   - Predicted density map
#   - Classification probabilities as bar chart
#   - True label vs predicted label
#
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import torch
import random

CLASS_NAMES  = {0: "Normal", 1: "Compressed", 2: "Turbulent"}
CLASS_COLORS = {0: "#2196F3", 1: "#FF9800", 2: "#F44336"}


def flow_to_rgb(flow_np):
    """
    Converts a 2-channel flow array (H,W,2) to an RGB image
    using HSV colour encoding: hue=direction, value=magnitude.
    """
    fx = flow_np[:, :, 0]
    fy = flow_np[:, :, 1]
    magnitude = np.sqrt(fx**2 + fy**2)
    angle     = np.arctan2(fy, fx)          # -pi to pi

    hsv = np.zeros((*magnitude.shape, 3), dtype=np.uint8)
    hsv[:, :, 0] = ((angle + np.pi) / (2 * np.pi) * 179).astype(np.uint8)
    mag_norm      = magnitude / (magnitude.max() + 1e-6)
    hsv[:, :, 1] = (mag_norm * 255).astype(np.uint8)
    hsv[:, :, 2] = 255

    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def visualise_predictions(model, dataset, n_samples=9, seed=42):
    """
    Visualises model predictions on n_samples test items.
    Samples are drawn to include all three classes (3 per class).
    """
    model.eval()
    random.seed(seed)

    # Gather indices per class
    class_indices = {0: [], 1: [], 2: []}
    for i, w in enumerate(dataset.windows):
        class_indices[w["label"]].append(i)

    # Sample 3 from each class
    selected = []
    for cls in range(3):
        chosen = random.sample(class_indices[cls],
                               min(n_samples // 3, len(class_indices[cls])))
        selected.extend(chosen)
    random.shuffle(selected)
    selected = selected[:n_samples]

    n_cols = 3
    n_rows = len(selected)

    fig = plt.figure(figsize=(18, n_rows * 5))
    fig.patch.set_facecolor("#1a1a2e")

    gs_outer = gridspec.GridSpec(n_rows, 1, hspace=0.08,
                                 left=0.02, right=0.98,
                                 top=0.95, bottom=0.02)

    fig.suptitle(
        "CrowdNet — Per-Sample Predictions on Test Set",
        fontsize=18, color="white", fontweight="bold", y=0.98
    )

    with torch.no_grad():
        for row_idx, sample_idx in enumerate(selected):

            frames, flows, label, density_target = dataset[sample_idx]

            # Forward pass
            frames_in = frames.unsqueeze(0).to(device)
            flows_in  = flows.unsqueeze(0).to(device)
            logits, density_pred = model(frames_in, flows_in)
            density_pred = torch.clamp(density_pred, min=0.0)

            probs     = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
            pred_cls  = int(probs.argmax())
            true_cls  = int(label.item())
            correct   = pred_cls == true_cls

            # Extract last frame for display
            last_frame = frames[-1, 0].numpy()       # (H, W)

            # Extract last flow for display
            last_flow  = flows[-1].permute(1, 2, 0).numpy()   # (H, W, 2)
            flow_rgb   = flow_to_rgb(last_flow)

            # Density map
            dens_map   = density_pred.squeeze().cpu().numpy()  # (28, 28)

            # ── Inner grid: 4 panels per row ─────────────────
            gs_inner = gridspec.GridSpecFromSubplotSpec(
                1, 4, subplot_spec=gs_outer[row_idx],
                wspace=0.04, hspace=0
            )

            border_color = "#4CAF50" if correct else "#F44336"

            # ─── Panel 1: Last frame ─────────────────────────
            ax1 = fig.add_subplot(gs_inner[0])
            ax1.imshow(last_frame, cmap="gray", vmin=0, vmax=1)
            ax1.set_title(
                f"Frame (last)\nTrue: {CLASS_NAMES[true_cls]}",
                color="white", fontsize=9, pad=3
            )
            ax1.axis("off")
            for spine in ax1.spines.values():
                spine.set_edgecolor(border_color)
                spine.set_linewidth(2)

            # ─── Panel 2: Optical flow ───────────────────────
            ax2 = fig.add_subplot(gs_inner[1])
            ax2.imshow(flow_rgb)
            ax2.set_title(
                "Farneback Flow\n(hue=direction, brightness=magnitude)",
                color="white", fontsize=9, pad=3
            )
            ax2.axis("off")

            # ─── Panel 3: Density map ────────────────────────
            ax3 = fig.add_subplot(gs_inner[2])
            im = ax3.imshow(dens_map, cmap="hot", interpolation="bilinear")
            pred_count = float(dens_map.sum())
            true_count = float(density_target.sum().item())
            ax3.set_title(
                f"Predicted Density\nPred cnt: {pred_count:.0f}  True cnt: {true_count:.0f}",
                color="white", fontsize=9, pad=3
            )
            ax3.axis("off")
            plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)

            # ─── Panel 4: Class probabilities ────────────────
            ax4 = fig.add_subplot(gs_inner[3])
            ax4.set_facecolor("#16213e")
            colors = [CLASS_COLORS[i] for i in range(3)]
            bars   = ax4.barh(
                [CLASS_NAMES[i] for i in range(3)],
                probs, color=colors, height=0.5
            )
            ax4.set_xlim(0, 1)
            ax4.set_xlabel("Probability", color="white", fontsize=8)
            ax4.tick_params(colors="white", labelsize=8)
            ax4.spines["bottom"].set_color("white")
            ax4.spines["left"].set_color("white")
            ax4.spines["top"].set_visible(False)
            ax4.spines["right"].set_visible(False)
            for bar, prob in zip(bars, probs):
                ax4.text(
                    prob + 0.02, bar.get_y() + bar.get_height() / 2,
                    f"{prob:.3f}", va="center", color="white", fontsize=8
                )
            verdict = "CORRECT" if correct else "WRONG"
            verdict_col = "#4CAF50" if correct else "#F44336"
            ax4.set_title(
                f"Pred: {CLASS_NAMES[pred_cls]}  [{verdict}]",
                color=verdict_col, fontsize=9, fontweight="bold", pad=3
            )

    plt.savefig(
        "crowdnet_predictions_new.png",
        dpi=150, bbox_inches="tight",
        facecolor="#1a1a2e"
    )
    plt.show()
    print("Saved: crowdnet_predictions_new.png")


# ── Run visualisation ─────────────────────────────────────────
visualise_predictions(model, test_dataset, n_samples=9, seed=42)


# ============================================================
# ADDITIONAL: Temporal evolution plot for one sequence
# ============================================================

def visualise_temporal_evolution(model, dataset, class_target=1, seed=7):
    """
    Shows how the model's confidence evolves frame by frame
    across a single sequence, for one sequence of class_target.
    Picks 6 evenly-spaced frames from the sequence window.
    """
    model.eval()
    random.seed(seed)

    # Find a window of the target class
    candidates = [i for i, w in enumerate(dataset.windows)
                  if w["label"] == class_target]
    idx = random.choice(candidates)
    frames, flows, label, density_target = dataset[idx]

    T = frames.shape[0]
    step_indices = np.linspace(0, T - 1, 6, dtype=int)

    fig, axes = plt.subplots(3, 6, figsize=(22, 9))
    fig.patch.set_facecolor("#1a1a2e")
    fig.suptitle(
        f"Temporal Evolution — True Class: {CLASS_NAMES[int(label.item())]}",
        fontsize=14, color="white", fontweight="bold"
    )

    with torch.no_grad():
        for col, t in enumerate(step_indices):

            # Run model up to timestep t (truncated sequence)
            frames_in = frames[:t+1].unsqueeze(0).to(device)
            flows_in  = flows[:t+1].unsqueeze(0).to(device)
            logits, _  = model(frames_in, flows_in)
            probs      = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
            pred_cls   = int(probs.argmax())

            # Frame
            ax_frame = axes[0, col]
            ax_frame.imshow(frames[t, 0].numpy(), cmap="gray", vmin=0, vmax=1)
            ax_frame.set_title(f"t={t}", color="white", fontsize=9)
            ax_frame.axis("off")

            # Flow
            ax_flow = axes[1, col]
            flow_np = flows[t].permute(1, 2, 0).numpy()
            ax_flow.imshow(flow_to_rgb(flow_np))
            ax_flow.axis("off")

            # Probabilities
            ax_prob = axes[2, col]
            ax_prob.set_facecolor("#16213e")
            colors = [CLASS_COLORS[i] for i in range(3)]
            ax_prob.bar(
                [CLASS_NAMES[i] for i in range(3)],
                probs, color=colors
            )
            ax_prob.set_ylim(0, 1)
            ax_prob.tick_params(colors="white", labelsize=7)
            ax_prob.set_facecolor("#16213e")
            for spine in ax_prob.spines.values():
                spine.set_color("white")
            ax_prob.set_title(
                f"Pred: {CLASS_NAMES[pred_cls]}",
                color=CLASS_COLORS[pred_cls], fontsize=8, fontweight="bold"
            )

    axes[0, 0].set_ylabel("Frame", color="white", fontsize=9)
    axes[1, 0].set_ylabel("Flow (HSV)", color="white", fontsize=9)
    axes[2, 0].set_ylabel("Class Prob.", color="white", fontsize=9)

    plt.tight_layout()
    plt.savefig(
        "crowdnet_temporal_evolution_new.png",
        dpi=150, bbox_inches="tight",
        facecolor="#1a1a2e"
    )
    plt.show()
    print("Saved: crowdnet_temporal_evolution_new.png")


# Show temporal evolution for one Bottleneck sequence
#visualise_temporal_evolution(model, test_dataset, class_target=1, seed=7)
# Show temporal evolution for one Panic sequence
visualise_temporal_evolution(model, test_dataset, class_target=2, seed=13)

C:\Users\RAJESH\AppData\Local\Temp\ipykernel_10628\1379269747.py:181: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved: crowdnet_predictions_new.png
Saved: crowdnet_temporal_evolution_new.png


C:\Users\RAJESH\AppData\Local\Temp\ipykernel_10628\1379269747.py:268: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
